# 实验七 · 综合实训：检测算法推理部署

**所属**：《并行计算》第七章 · AscendCL 应用开发　|　**难度**：⭐⭐⭐⭐ 综合　|　**预计时长**：60~80 分钟

本实验把一个真实的目标检测模型**从权重文件完整部署为可运行的推理程序**，并回答一个问题: **一个推理应用的时间都花在哪里？把预处理交给设备去做，可以省下多少时间？**

部署的链路是固定的四步：**PyTorch 权重 → ONNX → `.om` → AscendCL 程序**。在最后一步上，本实验写两个版本。**两版读同一份 letterbox 好的图片、产生同样的检出结果**，唯一的区别是**预处理三步转换（转 FLOAT、除以 255、HWC 换成 CHW）由谁来做**：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本 | 三步转换由谁做 | 主机送出去的数据 | 要观察的现象 |
| --- | --- | --- | --- |
| <strong>v1 主机侧预处理</strong> | 主机 CPU，几行 C++ 循环 | FLOAT 张量，4915200 字节 | 这三步在主机上要多久 |
| <strong>v2 AIPP 预处理</strong> | <strong>AIPP，在模型内部</strong> | <strong>UINT8 图像，1228800 字节</strong> | 预处理这一段快了多少，代价是什么 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">三步转换由谁做</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">主机送出去的数据</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要观察的现象</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v1 主机侧预处理</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机 CPU，几行 C++ 循环</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">FLOAT 张量，4915200 字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">这三步在主机上要多久</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>v2 AIPP 预处理</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>AIPP，在模型内部</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>UINT8 图像，1228800 字节</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">预处理这一段快了多少，代价是什么</td>
</tr>
</tbody>
</table>

**读图与 letterbox 由 OpenCV 完成，两版共用**（§5.3），因此这两步的耗时对两版是同一个常数，不进入对照。

> **实验说明**
> 1. 本实验的核心内容有三点：**部署链路的四步**、**端到端的阶段分解**、**用 AIPP 做预处理比在主机上做快多少、代价是什么**。
> 2. **本实验用真实的 YOLOv13 检测模型与真实图片**，因此需要联网，并需要 `torch` 与 YOLOv13 自带的 `ultralytics` 分支把权重导成 ONNX。这一步在 Notebook 里完成。
> 3. 请自上而下依次执行各单元格（Shift+Enter）。耗时最长的是依赖安装那一格；ONNX 导出在数秒量级，两次 ATC 转换各在一分钟量级。
> 4. 本实验依赖 **CANN 9.0.0 及以上**与 **Atlas A2/A3 训练推理系列产品**，并需要 `atc` 命令可用。
> 5. 本实验综合运用前面各节的内容，**建议按课程顺序学习**。


## 🎯 学习目标

完成本实验后，学生应能够：

- 说出把一个检测模型部署到昇腾平台的完整链路，并逐步执行下来
- 把 PyTorch 权重导成 ONNX、核对它的算子集与文件格式版本，再用 ATC 转出带 AIPP 的 `.om`
- 把一张任意尺寸的图片处理成检测模型要的输入：letterbox 缩放与补边，并说明它为什么不能直接拉伸
- 从检测模型的原始输出里解出候选框，并用 NMS 去掉重复框
- 把一个推理应用的端到端时间拆成预处理、推理、后处理三段，并说出瓶颈在哪一侧
- 说明把预处理搬到设备上的两条路——媒体数据处理算子与 AIPP——各自的形态与适用场合
- 说出 AIPP 承担的是哪几步预处理，以及把它们从主机移进模型之后，时间与传输量各有什么变化
- 设计一次受控的性能对照：说出哪一个量在变，其余的量各用什么办法按住
- 说明 AIPP 与媒体处理算子的分工，写出一份静态 AIPP 配置，并列出静态与动态 AIPP 的差别
- 把检测框映射回原图并画出来，据此判断整条部署链路是否正确


## 🗺️ 学习路径

1. **问题的形状**：一个检测应用的端到端时间分成哪三段，为什么要先分解再动手
2. **两条路**：把预处理搬到设备上有两种做法——媒体数据处理算子与 AIPP，它们的分工与适用场合
3. **AIPP**：它把哪几步编进了模型，静态与动态两种形态的差别在哪里
4. **素材准备**：把权重导成 ONNX、核对它的版本、用 OpenCV 把图片 letterbox，再转出带与不带 AIPP 的两个 `.om`
5. **程序实现**：两版共用一份代码，只在那三步转换由谁做上分岔
6. **测量与核对**：把端到端拆成三段分别测量，把检测框画回原图核对整条链路


## 1. 一个检测应用的时间都花在哪

一个真实的检测应用，时间分在三段上：

$$\text{读图与预处理} \;\to\; \text{推理} \;\to\; \text{后处理与输出}$$

第一段要把原始图像变成模型要的张量：读图、letterbox、转 FLOAT、归一化、排布转换。第三段要把模型的输出变成有意义的结果，本实验是解框加非极大值抑制。

**最简单的实现是这两段都在主机上。** 于是设备在等主机：它算完一张图，要等主机准备好下一张。

### 1.1 本实验要回答的两个问题

**第一个问题：三段的比例是多少。** 若预处理占了七成，那么优化推理带来的改善十分有限。

**第二个问题：把预处理交给 AIPP，比在主机上做快多少。** 预处理里有三步是逐像素的算术——**转 FLOAT、除以 255、HWC 换成 CHW**。它们可以在主机上用 CPU 做，也可以在模型转换时写进 `.om`、由 **AIPP** 在设备上随推理一起做。本实验把两条路都实现出来，其余条件完全一致：

- 读图与 letterbox 都由 OpenCV 在主机上完成，**两版共用同一份产物**（§5.3）；
- 两版检出的结果应当一致（§10）；
- 差别只有那三步转换做在哪一侧，以及由此决定的**送进设备的数据类型**。

这一个变量同时改变两件事：

<!--
| | v1 主机侧预处理 | v2 AIPP 预处理 |
| --- | --- | --- |
| 主机的计算 | 逐像素三步转换，$640\times640\times3$ 个元素 | **无** |
| 主机送出去的数据 | FLOAT，4915200 字节 | **UINT8，1228800 字节** |
| 这三步在哪里做 | 主机 CPU | **设备，随推理一起** |
-->

<table>
  <thead>
    <tr>
      <th style="text-align: left;"></th>
      <th style="text-align: left;">v1 主机侧预处理</th>
      <th style="text-align: left;">v2 AIPP 预处理</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">主机的计算</td>
      <td style="text-align: left;">逐像素三步转换，640×640×3 个元素</td>
      <td style="text-align: left;"><strong>无</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">主机送出去的数据</td>
      <td style="text-align: left;">FLOAT，4915200 字节</td>
      <td style="text-align: left;"><strong>UINT8，1228800 字节</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">这三步在哪里做</td>
      <td style="text-align: left;">主机 CPU</td>
      <td style="text-align: left;"><strong>设备，随推理一起</strong></td>
    </tr>
  </tbody>
</table>

### 1.2 本实验不测什么

- **不训练也不评估模型。** 本实验用现成的 YOLOv13 权重，只核对它在一张已知图片上检出的结果对不对，不做精度评测。
- **不做检测算法本身。** letterbox 与 NMS 讲清楚够用即可，检测网络的结构与训练属于计算机视觉课程。
- **不做视频。** 平台边界见 §2.4：本实验所用产品不支持视频编码输出。


## 2. 把预处理搬到设备上的两条路

预处理不一定要在主机上做。CANN 提供了两条把它挪到设备侧的路，**形态完全不同**，本实验走的是第二条。

### 2.1 第一条路：媒体数据处理算子

媒体数据处理是一整套在专用硬件单元上完成的图像与视频功能。《应用开发指南》给出的全景是：

<img src="images/07.07_dvpp.png" alt="图像/视频数据处理" width="900">

这些功能以**算子**的形式提供，由应用程序显式调用。它们的接口形态与内置的 aclnn 算子完全一致：**两段式，第一段在主机侧做校验与切块并给出 workspace 大小，第二段把核函数下发到 Stream**：

```cpp
acldvppStatus acldvppResizeGetWorkspaceSize(const aclTensor* self, uint32_t interpolationMode,
                                            aclTensor* out, uint64_t* workspaceSize,
                                            aclOpExecutor** executor);
acldvppStatus acldvppResize(void* workspace, uint64_t workspaceSize,
                            aclOpExecutor* executor, aclrtStream stream);
```

《应用开发指南》给出的调用流程是：

<img src="images/07.07_two_phase.png" alt="单算子调用的接口调用流程" height="620">

图上把接口名写成 `acl*xxXxx*GetWorkspaceSize` 与 `acl*xxXxx*` 这样的通配形式，右侧竖排的标题也写作「单算子调用」：**这一套流程对整类单算子成立**。图中蓝色为必选步骤，绿色为可选步骤；虚线框内是一次调用的完整形态——申请内存、传输数据、算出 workspace 大小并申请、执行算子、同步等待、释放内存。

**代价是它要写进应用程序**：额外的库、额外的张量描述符、额外的 workspace 管理，以及一次显式的流同步。

### 2.2 这一类算子的三个代表与它们的约束

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 算子 | 做的事 | 关键约束 |
| --- | --- | --- |
| <code>acldvppResize</code> | 缩放 | 输入 UINT8/FLOAT，Format 支持 NCHW/NHWC，<strong>N 为 1 或空、C 为 1 或 3</strong>；输出的 dataType 与 Format 必须与输入一致，N 轴与 C 轴大小也要一致 |
| <code>acldvppImgToTensor</code> | UINT8 转 FLOAT | 输入必须是 UINT8，<strong>输出的 Format 与 Shape 必须与输入完全一致</strong>——它只换数据类型，不换排布 |
| <code>acldvppNormalize</code> | 按通道减均值除标准差 | <strong>输入必须是 NHWC</strong>；均值与标准差用 <code>aclCreateFloatArray</code> 创建、数据放在主机侧，<strong>size 必须等于 C 轴大小</strong> |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">算子</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">做的事</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">关键约束</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>acldvppResize</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">缩放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入 UINT8/FLOAT，Format 支持 NCHW/NHWC，<strong>N 为 1 或空、C 为 1 或 3</strong>；输出的 dataType 与 Format 必须与输入一致，N 轴与 C 轴大小也要一致</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>acldvppImgToTensor</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">UINT8 转 FLOAT</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入必须是 UINT8，<strong>输出的 Format 与 Shape 必须与输入完全一致</strong>——它只换数据类型，不换排布</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><code>acldvppNormalize</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">按通道减均值除标准差</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>输入必须是 NHWC</strong>；均值与标准差用 <code>aclCreateFloatArray</code> 创建、数据放在主机侧，<strong>size 必须等于 C 轴大小</strong></td>
</tr>
</tbody>
</table>

### 2.3 第二条路：AIPP

AIPP 把同样这几步**编进模型**，由 ATC 在模型转换时完成，运行期一行代码都不用写。它没有上面那些工程代价：不加库、不加接口调用、不加内存管理。**本实验使用该方案**，§3 详细讲它。

**两条路的适用场合不同。** 算子在运行期调用，每次都可以不一样，适合按图变化的处理；AIPP 在转换时定死，适合每张图都一样的固定处理。本实验的三步转换属于后者。

### 2.4 平台边界：本课程所用产品支持哪些媒体处理能力

一个方案可不可行，先看产品支不支持。**本课程所用产品的媒体处理能力有明确的边界：**

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 能力 | 本课程所用产品是否支持 | 对本实验的影响 |
| --- | --- | --- |
| VPC（图像处理：缩放、抠图、色域转换等） | <strong>支持</strong> | v2 的预处理链可行 |
| JPEGD（JPEG 解码） | <strong>支持</strong> | 可以直接送 JPEG 进去，本实验用原始像素以免依赖图片文件 |
| JPEGE（JPEG 编码） | <strong>支持</strong> | 结果图可以在设备上编码后回传 |
| PNGD（PNG 解码） | <strong>支持</strong> | 同 JPEGD |
| VDEC（视频解码） | <strong>支持</strong> | 视频输入可行 |
| <strong>VENC（视频编码）</strong> | <strong>不支持</strong> | <strong>不能设计视频编码输出的环节</strong> |
| Camera 场景、NVR 场景 | <strong>不支持</strong> | 不能从摄像头直接取流 |
| 音频获取与播放 | <strong>不支持</strong> | 音频类应用要另找方案 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">能力</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">本课程所用产品是否支持</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">对本实验的影响</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">VPC（图像处理：缩放、抠图、色域转换等）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>支持</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">v2 的预处理链可行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">JPEGD（JPEG 解码）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>支持</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">可以直接送 JPEG 进去，本实验用原始像素以免依赖图片文件</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">JPEGE（JPEG 编码）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>支持</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">结果图可以在设备上编码后回传</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">PNGD（PNG 解码）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>支持</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">同 JPEGD</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">VDEC（视频解码）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>支持</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">视频输入可行</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>VENC（视频编码）</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不支持</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不能设计视频编码输出的环节</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">Camera 场景、NVR 场景</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不支持</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不能从摄像头直接取流</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">音频获取与播放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不支持</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">音频类应用要另找方案</td>
</tr>
</tbody>
</table>

**因此本实验只做静态图片的输入与数值结果的输出**，不设计视频编码输出的环节。

## 3. AIPP：另一条把预处理并进模型的路

AIPP（AI Preprocessing）是把色域转换、减均值乘系数、抠图、补边这些操作**并进模型**的机制。它在模型转换时由 `--insert_op_conf` 配置。

### 3.1 与媒体处理算子的分工

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
|  | 媒体处理算子 | AIPP |
| --- | --- | --- |
| 在哪里执行 | 专用的媒体处理硬件单元 | **AI Core 上**，作为模型的一部分 |
| 什么时候确定 | 运行期，每次调用都可以不同 | 静态 AIPP 在模型转换时确定；动态 AIPP 运行期可改 |
| 调用方式 | 两段式算子，显式下发 | **不用调用**，模型执行时自动完成 |
| 能做什么 | 解码、缩放、抠图、归一化、翻转、仿射等 | 色域转换、减均值乘系数、抠图、补边 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"></th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">媒体处理算子</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">AIPP</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">在哪里执行</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">专用的媒体处理硬件单元</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>AI Core 上</strong>，作为模型的一部分</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">什么时候确定</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">运行期，每次调用都可以不同</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">静态 AIPP 在模型转换时确定；动态 AIPP 运行期可改</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">调用方式</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两段式算子，显式下发</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>不用调用</strong>，模型执行时自动完成</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">能做什么</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">解码、缩放、抠图、归一化、翻转、仿射等</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">色域转换、减均值乘系数、抠图、补边</td>
</tr>
</tbody>
</table>

### 3.2 静态与动态 AIPP

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
|  | 静态 AIPP | 动态 AIPP |
| --- | --- | --- |
| 参数何时确定 | 模型转换时固化在 `.om` 里 | **运行期设置** |
| 模型的输入个数 | 不变 | **多出一个输入**，用于承载 AIPP 配置 |
| 多 Batch | 各 Batch 共用同一份参数 | **每个 Batch 可以各不相同** |
| 改参数的代价 | 要重新转换模型 | 改一次调用即可 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"></th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">静态 AIPP</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">动态 AIPP</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参数何时确定</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">模型转换时固化在 <code>.om</code> 里</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>运行期设置</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">模型的输入个数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">不变</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>多出一个输入</strong>，用于承载 AIPP 配置</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">多 Batch</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">各 Batch 共用同一份参数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>每个 Batch 可以各不相同</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">改参数的代价</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">要重新转换模型</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">改一次调用即可</td>
</tr>
</tbody>
</table>

动态 AIPP 的接口调用次序由《应用开发指南》给出：

<img src="images/07.07_aipp.png" alt="动态 AIPP 的接口调用流程" height="560">

图中「设置动态 AIPP 参数值」这一步包含若干个设置接口，其中 **`aclmdlSetAIPPSrcImageSize` 是必调的**，漏掉会在执行时出错。本实验用的是静态 AIPP，不涉及该链条。

### 3.3 约束必须逐条记住

1. **静态 AIPP 与动态 AIPP 不能同时配置。**
2. **AIPP（含静态）与动态维度（ND）不能同时使用。**
3. 动态 AIPP 与动态 Batch 同用时，`aclmdlCreateAIPP` 的 `batchSize` 要填**最大**的那一档。
4. 动态 AIPP 若开启了抠图、缩放或补边，**不能与动态分辨率同用**。
5. 动态 AIPP 的参数在每次推理前设置，用完要及时销毁 `aclmdlAIPP` 类型。

### 3.4 本实验用的那份配置

v2 用的模型带静态 AIPP，v1 用的不带。配置只有几行：

```
aipp_op {
    aipp_mode : static
    input_format : RGB888_U8
    src_image_size_w : 640
    src_image_size_h : 640
    csc_switch : false
    rbuv_swap_switch : false
    var_reci_chn_0 : 0.003921568627451
    var_reci_chn_1 : 0.003921568627451
    var_reci_chn_2 : 0.003921568627451
}
```

`var_reci_chn_*` 是 $1/255$，也就是 v1 在主机上做的那次归一化。**排布转换没有出现在配置里**：它是 AIPP 的固有行为，模型输入声明成 NCHW 时，AIPP 就把 NHWC 的图像转成 NCHW 送进去。

于是 v1 与 v2 只有中间三步不同：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 步骤 | v1 主机侧预处理 | v2 AIPP 预处理 |
| --- | --- | --- |
| 读图与 letterbox 到 640 | OpenCV（主机） | OpenCV（主机），**同一份产物** |
| **转 FLOAT** | **主机 C++ 循环** | **AIPP** |
| **除以 255** | **主机 C++ 循环** | **AIPP** |
| **HWC 换成 CHW** | **主机 C++ 循环** | **AIPP** |
| 主机送出去的数据 | FLOAT，4915200 字节 | **UINT8，1228800 字节** |
| 用哪个模型 | `yolov13.om` | `yolov13_aipp.om` |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
  <thead>
    <tr>
      <th style="text-align: left;">步骤</th>
      <th style="text-align: left;">v1 主机侧预处理</th>
      <th style="text-align: left;">v2 AIPP 预处理</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">读图与 letterbox 到 640</td>
      <td style="text-align: left;">OpenCV（主机）</td>
      <td style="text-align: left;">OpenCV（主机），<strong>同一份产物</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>转 FLOAT</strong></td>
      <td style="text-align: left;"><strong>主机 C++ 循环</strong></td>
      <td style="text-align: left;"><strong>AIPP</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>除以 255</strong></td>
      <td style="text-align: left;"><strong>主机 C++ 循环</strong></td>
      <td style="text-align: left;"><strong>AIPP</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>HWC 换成 CHW</strong></td>
      <td style="text-align: left;"><strong>主机 C++ 循环</strong></td>
      <td style="text-align: left;"><strong>AIPP</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">主机送出去的数据</td>
      <td style="text-align: left;">FLOAT，4915200 字节</td>
      <td style="text-align: left;"><strong>UINT8，1228800 字节</strong></td>
    </tr>
    <tr>
      <td style="text-align: left;">用哪个模型</td>
      <td style="text-align: left;"><code>yolov13.om</code></td>
      <td style="text-align: left;"><code>yolov13_aipp.om</code></td>
    </tr>
  </tbody>
</table>

第一步与最后的检出结果两版相同；**中间三步是同一件事，只是做在了不同的地方**，而做在哪里又决定了主机要送出去多少字节。

**两条路的检出结果必须一致**。


## 4. 环境准备与检查

先建目录、导入 CANN 环境变量。


In [ ]:
!mkdir -p src_detect model data

import os, subprocess

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))
print("🎉 环境变量导入完成")


本实验的程序只用到两类接口：**Runtime**（设备、Context、内存与传输）与**模型管理**（加载、执行、卸载），都声明在 `acl/acl.h` 里。链接的规则是**程序包含了哪些头文件，就链接对应的库文件**，因此链接 `libacl_rt.so` 与 `libacl_mdl.so` 即可；部分安装形态把两者合并在 `libascendcl.so` 中，下一格按候选名逐个探测。


In [ ]:
import os, shutil, sys

print("atc      :", shutil.which("atc") or "⚠️  未找到")
print("g++      :", shutil.which("g++") or "⚠️  未找到")

ascend_home = os.environ.get("ASCEND_HOME_PATH", "")
inc_root = f"{ascend_home}/include"
ACL_INCDIRS = ["-I" + inc_root] if os.path.isdir(inc_root) else []
ok = os.path.exists(os.path.join(inc_root, "acl/acl.h"))
print(f"头文件 acl/acl.h                      -> {'找到' if ok else '⚠️  未找到'}")

lib_dirs = [
    path
    for path in (f"{ascend_home}/lib64", f"{ascend_home}/devlib")
    if os.path.isdir(path)
]
ACL_LIBDIRS = ["-L" + path for path in lib_dirs]


def find_lib(candidates):
    "按候选名找第一个存在的库；部分安装形态把多类接口合并在一个库里。"
    for name in candidates:
        for lib_dir in lib_dirs:
            if os.path.exists(os.path.join(lib_dir, f"lib{name}.so")):
                return name
    return None


selected = []
for purpose, candidates in [
    ("Runtime", ["acl_rt", "ascendcl"]),
    ("模型管理", ["acl_mdl", "ascendcl"]),
]:
    name = find_lib(candidates)
    print(f"{purpose:<10} 候选 {candidates} -> {name}")
    if name is not None and name not in selected:
        selected.append(name)
ACL_LIBS = ["-l" + name for name in selected]
print("头文件搜索路径 :", " ".join(ACL_INCDIRS))
print("链接参数      :", " ".join(ACL_LIBS))

for extra in os.environ.get("PYTHONPATH", "").split(":"):
    if extra and extra not in sys.path:
        sys.path.append(extra)
SOC_VERSION = "Ascend910B1"  # ← 读不出来时的回退值，请按本机型号修改
try:
    import acl

    acl.init()
    SOC_VERSION = acl.get_soc_name()
    acl.finalize()
except Exception as exc:
    print("⚠️  pyACL 读取型号失败，使用回退值：", exc)
print("设备型号      :", SOC_VERSION)


§5 要把 YOLOv13 的 PyTorch 权重导成 ONNX，这一步要用 `torch` 与 YOLOv13 自带的 `ultralytics` 分支。**标准版 `ultralytics` 加载不了 YOLOv13 的权重**：它用了 `DSC3k2` 一类自定义模块，只有作者仓库里的那份分支能够识别。

下面这一格把依赖装齐。在 aarch64 上这一格通常是整个实验里最耗时的一段，请留出时间。

**分支是用 `--no-deps` 装的**——否则 pip 在解析依赖时可能顺带换掉本机已经适配好的 `torch`。代价是 `ultralytics` 自己的依赖要手工补齐。


In [ ]:
import importlib, importlib.util, os, subprocess, sys


def have(module):
    "这个包在不在。只查找不执行——torch 在 Notebook 内核里恰好是导入不起来的那一个。"
    try:
        return importlib.util.find_spec(module) is not None
    except (ImportError, ValueError):
        return False


def pip_install(packages):
    "调一次 pip 装若干个包，打印它最后两行有用的输出；返回是否成功。"
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet"] + packages,
        capture_output=True,
        text=True,
    )
    # pip 自身的版本提示与本次安装无关，滤掉，免得把真正的报错顶出屏幕
    NOISE = ("new release of pip", "pip install --upgrade")
    lines = [
        line
        for line in (proc.stdout + proc.stderr).strip().splitlines()
        if line.strip() and not any(word in line for word in NOISE)
    ]
    for line in lines[-2:]:
        print("   " + line[:140])
    if proc.returncode != 0:
        print("   ❌ pip 返回 %d" % proc.returncode)
    return proc.returncode == 0


def ensure(title, pairs):
    "逐个检查 (import 名, 包名)，缺的合成一次 pip 调用装上。返回仍然缺失的包名。"
    ready = [module for module, _ in pairs if have(module)]
    missing = [package for module, package in pairs if not have(module)]
    print("%-12s 已就绪 %s" % (title, ready if ready else "（无）"))
    if not missing:
        return []
    print("%-12s 缺失 %s，开始安装" % ("", missing))
    pip_install(missing)
    # 刚装好的包在解释器启动之后才出现，要先让查找器丢掉缓存的目录列表
    importlib.invalidate_caches()
    still = [package for module, package in pairs if not have(module)]
    print("%-12s 安装后仍缺 %s" % ("", still if still else "（无）"))
    return still


# 第一组：本 Notebook 的 Python 单元格自己要用的。onnx 供 §5.2 与 §6.1 读版本号，
# matplotlib 供 §11 画框。
_ = ensure(
    "基础库",
    [
        ("numpy", "numpy"),
        ("PIL", "Pillow"),
        ("matplotlib", "matplotlib"),
        ("onnx", "onnx"),
    ],
)

# 第二组：把权重导成 ONNX 这条路要用的，只有 torch（追踪导出）。onnx 上一组已经查过。
# 本实验的导出关掉了 simplify（§5.1），因此不需要 onnxslim；导出后端固定为传统追踪
# 导出，因此也不需要 onnxscript。
EXPORT_MISSING = ensure("导出工具链", [("torch", "torch")])
if EXPORT_MISSING:
    print("   ❌ 缺少它，§5.1 的导出无法进行：", EXPORT_MISSING)

# 第三组：YOLOv13 的 ultralytics 分支。仓库源码里带一份，装它之前要先卸掉标准版。
YOLO_REPO = "yolov13-main"
if have("ultralytics") and os.path.isdir(YOLO_REPO):
    print("%-12s 已就绪（YOLOv13 分支）" % "ultralytics")
else:
    print("%-12s 准备安装 YOLOv13 分支" % "ultralytics")
    if not os.path.isdir(YOLO_REPO):
        subprocess.run(
            [
                "curl",
                "-fsSL",
                "-o",
                "yolov13.zip",
                "https://codeload.github.com/iMoonLab/yolov13/zip/refs/heads/main",
            ],
            check=False,
        )
        subprocess.run(["unzip", "-q", "-o", "yolov13.zip"], check=False)
    if os.path.isdir(YOLO_REPO):
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "ultralytics"],
            capture_output=True,
        )
        pip_install(["--no-deps", YOLO_REPO + "/"])
    else:
        print("   ❌ 没有取到 YOLOv13 源码，请检查网络后重试")

# 第四组：分支是用 --no-deps 装的，它自己的依赖在这里补齐。这一组按 ultralytics
# 的依赖清单给出，已经装好的只报一行、不重复安装。
# OpenCV 用 headless 版：aarch64 上 opencv-python 会拉进 libGLdispatch。
_ = ensure(
    "分支的依赖",
    [
        ("yaml", "pyyaml"),
        ("cv2", "opencv-python-headless"),
        ("tqdm", "tqdm"),
        ("psutil", "psutil"),
        ("requests", "requests"),
        ("scipy", "scipy"),
        ("pandas", "pandas"),
        ("seaborn", "seaborn"),
        ("cpuinfo", "py-cpuinfo"),
        ("huggingface_hub", "huggingface_hub"),
        ("thop", "ultralytics-thop"),
    ],
)


## 5. 准备模型与图片

### 5.1 把 PyTorch 权重导成 ONNX

**部署链路的第一步。** YOLOv13 发布的是 PyTorch 权重（`.pt`），而 ATC 认的是 ONNX，中间这一步要自己做。`ultralytics` 提供了导出接口，一行即可：

```python
YOLO("model/yolov13n.pt").export(format="onnx", opset=11, imgsz=640)
```

两个参数要说明：`imgsz=640` 决定模型的输入边长，导出之后这个尺寸就固化在图中；`opset=11` 是算子集版本，**要选 ATC 支持的版本**，版本太新会在转换时报出不支持的算子。

导出得到的 `.onnx` 与权重同名、放在权重旁边。这一份图的 batch 维是常量 1，本实验只需要一张图一次推理，因此这样正合适。

In [ ]:
%%writefile model/export_onnx.py
# 导出脚本单独成文件：它要在带 LD_PRELOAD 的新进程里跑，写成文件比塞进 -c 更好读
import functools
import inspect

import torch
from ultralytics import YOLO

# 较新版本的 torch 默认改用 dynamo 后端导出，它不保证给出这里请求的 opset。
# 本实验要的是一份 ATC 能接受的图，因此把后端固定回传统的追踪导出。
# 旧版本的 torch 没有这个参数，此时下面这一段是空操作。
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    torch.onnx.export = functools.partial(torch.onnx.export, dynamo=False)
    print("导出后端   : 传统追踪导出（已显式指定 dynamo=False）")
else:
    print("导出后端   : 传统追踪导出（本机 torch 没有 dynamo 后端）")

# simplify 会调用 onnxslim 改写计算图。本实验不需要，关掉它少一个变量。
print(YOLO("model/yolov13n.pt").export(format="onnx", opset=11, imgsz=640, simplify=False))


In [ ]:
# 每一行 ! 都是独立的子 shell，因此 LD_PRELOAD 必须与 python3 写在同一条命令里
!test -s model/yolov13n.pt || wget -q -O model/yolov13n.pt "https://github.com/iMoonLab/yolov13/releases/download/yolov13/yolov13n.pt"

!SITE=$(python3 -c "import importlib.util, os; s = importlib.util.find_spec('torch'); print(os.path.dirname(os.path.dirname(s.origin)) if s and s.origin else '')"); \
 LIBGOMP=$(for p in "$SITE"/torch.libs/libgomp*.so* "$SITE"/torch/lib/libgomp*.so* /usr/lib/*/libgomp.so.1; do [ -e "$p" ] && echo "$p" && break; done); \
 echo "LD_PRELOAD : ${LIBGOMP:-（未找到 libgomp，直接导出）}"; \
 export LD_PRELOAD="$LIBGOMP" GLIBC_TUNABLES=glibc.rtld.optional_static_tls=2097152; \
 if [ -s model/yolov13n.onnx ]; then \
   echo "ONNX       : 已存在，跳过导出（要重导请先删掉 model/yolov13n.onnx）"; \
 else \
   python3 model/export_onnx.py; \
 fi

!ls -lh model/yolov13n.pt model/yolov13n.onnx 2>/dev/null

### 5.2 ONNX 的版本核对与规整

一份 `.onnx` 上有两个版本号，**它们各管一件事**：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 版本号 | 管什么 | 由谁决定 |
| --- | --- | --- |
| <strong>opset</strong>（算子集版本） | 图里用的是哪一版算子的语义 | 导出时请求，但**由导出后端最终决定** |
| <strong>IR version</strong>（文件格式版本） | 这个文件本身按哪一版 ONNX 规范存放 | 由本机 <code>onnx</code> 包的版本决定 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">版本号</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">管什么</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">由谁决定</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>opset</strong>（算子集版本）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">图里用的是哪一版算子的语义</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">导出时请求，但**由导出后端最终决定**</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>IR version</strong>（文件格式版本）</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">这个文件本身按哪一版 ONNX 规范存放</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">由本机 <code>onnx</code> 包的版本决定</td>
</tr>
</tbody>
</table>

超出范围时 ATC 的表现**不一定是一条清楚的错误**。解析器读到自己不认识的版本字段时可能异常退出，那时既没有错误码也没有提示信息。**这类失败不易定位，因为现象不指向模型文件本身。**

因此下面这一格在导出与转换之间加一道规整：**读出两个版本号，超出范围时改写为上限值，并重新做一次完整性校验。** 

**降低 opset 则未必可行。** ONNX 的版本转换要为每一个跨版本的算子准备一个适配器，缺一个就整体失败。因此本单元格对 opset 只做检查与提示：需要降低时，正确的做法是回到 §5.1 重新导出，而不是在此改写文件。


In [ ]:
import onnx

ONNX_PATH = "model/yolov13n.onnx"
# ATC 接受的上限：取本课程验证过的范围。换一个 CANN 版本要重新确认。
MAX_IR = 9
MAX_OPSET = 17


def onnx_versions(path):
    "读出一份 ONNX 的 IR version 与默认域的 opset。"
    model = onnx.load(path)
    opset = next(
        (o.version for o in model.opset_import if o.domain in ("", "ai.onnx")), None
    )
    return model, model.ir_version, opset


model, ir, opset = onnx_versions(ONNX_PATH)
print("导出得到  : IR version %s，opset %s" % (ir, opset))
print("ATC 上限  : IR version %d，opset %d" % (MAX_IR, MAX_OPSET))

if ir > MAX_IR:
    # 只改文件格式版本，不动图。改完立刻让校验器确认这份文件仍然自洽。
    model.ir_version = MAX_IR
    onnx.checker.check_model(model)
    onnx.save(model, ONNX_PATH)
    print("已把 IR version 压到 %d，并通过完整性校验" % MAX_IR)
else:
    print("IR version 在范围内，不作改动")

if opset is not None and opset > MAX_OPSET:
    print(
        "⚠️  opset 为 %d，高于上限 %d。这里不做降级（缺适配器时会整体失败）；"
        % (opset, MAX_OPSET)
    )
    print(
        "    正确的做法是删掉 %s 后回到 §5.1 重新导出——"
        "§5.1 已把导出后端固定为传统追踪导出，它会按请求给出 opset 11。" % ONNX_PATH
    )
else:
    print("opset 在范围内，不作改动")

_, ir, opset = onnx_versions(ONNX_PATH)
ready = ir <= MAX_IR and (opset is None or opset <= MAX_OPSET)
print("\n最终       : IR version %s，opset %s —— %s"
      % (ir, opset, "可以交给 ATC" if ready else "还不能交给 ATC"))


### 5.3 letterbox：把任意尺寸的图片变成模型要的输入

模型要的是 $640 \times 640$，图片却是任意尺寸的。直接拉伸会改变物体的长宽比，而模型在训练时没有见过这样形变的物体，检出的框在位置与尺寸上都会偏离。

**letterbox** 的做法是：按长边等比缩放，短边两侧补上灰边（$114,114,114$），凑成正方形。物体的比例不变，代价是画面里多了一些没有信息的边。

$$\text{scale} = \min\left(\frac{S}{W},\ \frac{S}{H}\right), \quad \text{pad}_x = \frac{S - W\cdot\text{scale}}{2}, \quad \text{pad}_y = \frac{S - H\cdot\text{scale}}{2}$$

**读图与 letterbox 由 OpenCV 完成，两个版本共用这一段，产物也是同一份文件**：一张 $640\times640$ 的 UINT8 图像，按 RGB 顺序、交错排布（HWC）存成 `data/bus_letterbox.bin`。**这是两版共同的起点**，此后 v1 与 v2 的差别才开始（§7.4）。

> ⚠️ **`cv2.imread` 读出来的通道顺序是 BGR，不是 RGB。** YOLOv13 训练时用的是 RGB，直接送 BGR 进去，检出的框会明显不对，而程序不会报任何错。这里用 `cv2.cvtColor` 显式转一次。
>
> AIPP 的配置里也有一个 `rbuv_swap_switch` 可以做这次交换。本实验把它放在 OpenCV 这一侧，**是为了让两个版本拿到完全相同的输入**。

缩放与补边的参数要记下来——§11 画框时要靠它们把坐标映射回原图。


In [ ]:
import os, subprocess

import cv2
import numpy as np

MODEL_SIDE = 640  # 模型的输入边长，letterbox 直接做到这个尺寸
CHANNELS = 3
PAD_VALUE = 114

IMAGE_URL = (
    "https://raw.githubusercontent.com/iMoonLab/yolov13/main/"
    "ultralytics/assets/bus.jpg"
)
IMAGE_PATH = "data/bus.jpg"
LETTERBOX_PATH = "data/bus_letterbox.bin"

if not os.path.exists(IMAGE_PATH):
    subprocess.run(["wget", "-q", "-O", IMAGE_PATH, IMAGE_URL], check=False)


def letterbox(jpg_path, side, bin_path):
    "OpenCV 读图，转成 RGB，等比缩放到 side×side 的画布中央，四周补灰。"
    bgr = cv2.imread(jpg_path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(jpg_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)  # ← 通道顺序必须转，见本节说明
    height, width = rgb.shape[:2]
    scale = min(side / width, side / height)
    new_w, new_h = int(round(width * scale)), int(round(height * scale))
    resized = cv2.resize(rgb, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((side, side, CHANNELS), PAD_VALUE, dtype="uint8")
    pad_x, pad_y = (side - new_w) // 2, (side - new_h) // 2
    canvas[pad_y : pad_y + new_h, pad_x : pad_x + new_w, :] = resized
    canvas.tofile(bin_path)
    return width, height, scale, pad_x, pad_y


ORIG_W, ORIG_H, SCALE, PAD_X, PAD_Y = letterbox(
    IMAGE_PATH, MODEL_SIDE, LETTERBOX_PATH
)
print("原图      : %s  %d × %d" % (IMAGE_PATH, ORIG_W, ORIG_H))
print(
    "letterbox : %s  %d × %d，%.1f KB"
    % (
        LETTERBOX_PATH,
        MODEL_SIDE,
        MODEL_SIDE,
        os.path.getsize(LETTERBOX_PATH) / 1024,
    )
)
print("缩放比    : %.6f，左右补 %d 像素，上下补 %d 像素" % (SCALE, PAD_X, PAD_Y))
print(
    "校验      : %d 字节 %s"
    % (
        os.path.getsize(LETTERBOX_PATH),
        "✅" if os.path.getsize(LETTERBOX_PATH) == MODEL_SIDE**2 * CHANNELS else "❌",
    )
)


## 6. 模型转换

**部署链路的第二步。** ATC 把 ONNX 编译成昇腾平台上可执行的 `.om`。**本实验转两个，区别只在带不带 AIPP**——这一对模型正是两个版本的区别所在：

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
|  | <code>yolov13.om</code>（v1 用） | <code>yolov13_aipp.om</code>（v2 用） |
| --- | --- | --- |
| 转换参数 | 只有 <code>--input_shape</code> | 另加 <code>--insert_op_conf</code> |
| 模型接收什么 | FLOAT，NCHW，每通道四个字节 | <strong>UINT8，NHWC，每通道一个字节</strong> |
| 转 FLOAT、除以 255、排布转换由谁做 | <strong>应用侧（主机 CPU）</strong> | <strong>AIPP，在模型内部</strong> |
| 主机要送过去的字节数 | 4915200 | <strong>1228800</strong> |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"></th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"><code>yolov13.om</code>（v1 用）</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;"><code>yolov13_aipp.om</code>（v2 用）</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">转换参数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">只有 <code>--input_shape</code></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">另加 <code>--insert_op_conf</code></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">模型接收什么</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">FLOAT，NCHW，每通道四个字节</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>UINT8，NHWC，每通道一个字节</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">转 FLOAT、除以 255、排布转换由谁做</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>应用侧（主机 CPU）</strong></td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>AIPP，在模型内部</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">主机要送过去的字节数</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">4915200</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;"><strong>1228800</strong></td>
</tr>
</tbody>
</table>

**两次转换用的是同一份 `yolov13n.onnx`**（§5.1），`--input_shape` 也完全一样。这一点值得留意：**AIPP 改变的是模型接收数据的方式，而不是模型本身**——转换时给 ATC 的输入形状仍按模型原本的声明写（`1,3,640,640`），AIPP 是加在图前面的一段，由它把送进来的 UINT8 NHWC 图像转成模型要的 FLOAT NCHW 张量。

**这张表就是本实验要比的东西**：同样一张 letterbox 好的图，三步转换放在应用侧还是放进模型，各要多少时间、各要传多少数据。

YOLOv13 的输入名是 `images`。每次转换在一分钟量级，`.om` 已存在时跳过。

### 6.1 转换之前先逐项核对模型

`--input_shape=images:1,3,640,640` 里的两样东西都是**从模型里读出来的，不是约定俗成的**：`images` 是输入的名字，`1,3,640,640` 是它的形状。**两者只要有一个与图里的对不上，转换就会失败**，而报出来的话未必直接指向这里。

因此下面这一格在调用 ATC 之前先核对以下几项：

- **输入的名字与形状**：直接与 `--input_shape` 对照；
- **opset 与 IR version**：即 §5.2 那两个版本号，这里再确认一次；
- **输出的形状**：应当是 $(1, 84, 8400)$，与 §7.2 的后处理读法一致。

**本单元格是一道前置校验，不只是一条提示**：只要有一项不合格就不调用 ATC。理由在 §5.2 说过——版本号超出范围时，ATC 未必给出清楚的错误信息，也可能异常退出。

### 6.2 转换失败时如何定位

ATC 的日志很长，**关键的那一行不一定在末尾**：它通常先报出根因（形如 `E19999`、`EZ3003` 的错误码行），再跟一段调用栈与一句收尾提示。因此下面这一格失败时做两件事：**把带错误码的行单独挑出来打印**，并把完整日志写进 `model/atc_<名字>.log`。

**若日志里一条错误码都没有**，只有一行 `Segmentation fault`，说明 ATC 在给出错误信息之前就已经退出。此时先回到 §5.2 核对两个版本号。


In [ ]:
import os, re, subprocess, time

# ONNX_PATH、MAX_IR、MAX_OPSET 都在 §5.2 那一格定义，这里沿用
INPUT_NAME = "images"  # ← 模型里那个输入的真名，下面这一步会核对
AIPP_CONF = "model/aipp_rgb.cfg"

with open(AIPP_CONF, "w", encoding="utf-8") as handle:
    handle.write(
        "aipp_op {\n"
        "    aipp_mode : static\n"
        "    input_format : RGB888_U8\n"
        "    src_image_size_w : %d\n"
        "    src_image_size_h : %d\n"
        "    csc_switch : false\n"
        "    rbuv_swap_switch : false\n"
        "    var_reci_chn_0 : 0.003921568627451\n"
        "    var_reci_chn_1 : 0.003921568627451\n"
        "    var_reci_chn_2 : 0.003921568627451\n"
        "}\n" % (MODEL_SIDE, MODEL_SIDE)
    )
print("AIPP 配置 :", AIPP_CONF)


def shape_of(value_info):
    "取一个张量的形状；动态维返回它的符号名。"
    return [
        (d.dim_param or d.dim_value)
        for d in value_info.type.tensor_type.shape.dim
    ]


def describe_onnx(path):
    "转换之前逐项核对模型。任何一项不合格都返回 False，ATC 就不会被调用（§6.1）。"
    try:
        import onnx
    except ImportError:
        print("❌ 没有 onnx 包，无法核对；请先运行 §4 的依赖检查")
        return False
    model = onnx.load(path)
    opset = next(
        (o.version for o in model.opset_import if o.domain in ("", "ai.onnx")), None
    )
    print("IR version :", model.ir_version)
    print(
        "opset      :",
        ", ".join(
            "%s=%d" % (o.domain or "ai.onnx", o.version) for o in model.opset_import
        ),
    )
    weights = {t.name for t in model.graph.initializer}
    inputs = [i for i in model.graph.input if i.name not in weights]
    for item in inputs:
        print("输入       : %-10s %s" % (item.name, shape_of(item)))
    for item in model.graph.output:
        print("输出       : %-10s %s" % (item.name, shape_of(item)))

    problems = []
    if model.ir_version > MAX_IR:
        problems.append(
            "IR version %d 高于上限 %d —— 请重新运行 §5.2 那一格"
            % (model.ir_version, MAX_IR)
        )
    if opset is not None and opset > MAX_OPSET:
        problems.append(
            "opset %d 高于上限 %d —— 请删掉 %s 后回到 §5.1 重新导出"
            % (opset, MAX_OPSET, path)
        )
    names = [item.name for item in inputs]
    if INPUT_NAME not in names:
        problems.append("INPUT_NAME 为 %r，而图里的输入是 %s，请按后者修改" % (INPUT_NAME, names))
    else:
        got = shape_of(inputs[names.index(INPUT_NAME)])
        if [d for d in got if isinstance(d, int)] != [1, CHANNELS, MODEL_SIDE, MODEL_SIDE]:
            problems.append(
                "图里的输入形状是 %s，与本实验要转的 %s 不一致"
                % (got, [1, CHANNELS, MODEL_SIDE, MODEL_SIDE])
            )
    for line in problems:
        print("❌ " + line)
    verdict = "全部合格，可以转换" if not problems else "有 %d 项不合格，本格不调用 ATC" % len(problems)
    print("核对结果   :", verdict)
    print()
    return not problems


def run_atc(name, extra):
    "调一次 atc。失败时挑出带错误码的行，并留下完整日志（§6.2）。"
    target = "model/%s.om" % name
    log_path = "model/atc_%s.log" % name
    if os.path.exists(target):
        print("✅ 已存在 %s（%.1f MB）\n" % (target, os.path.getsize(target) / 1048576))
        return True
    if not os.path.exists(ONNX_PATH):
        print("⚠️  缺少 %s，跳过 %s\n" % (ONNX_PATH, target))
        return False
    cmd = [
        "atc",
        "--model=" + ONNX_PATH,
        "--framework=5",
        "--output=model/" + name,
        "--soc_version=" + SOC_VERSION,
        "--output_type=FP32",
    ] + extra
    print("$ " + " ".join(cmd))
    start = time.time()
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True)
    except FileNotFoundError:
        print("❌ 找不到 atc 命令，请确认 §4 的环境变量已导入\n")
        return False
    log = proc.stdout + proc.stderr
    with open(log_path, "w", encoding="utf-8") as handle:
        handle.write(log)

    if os.path.exists(target):
        print(
            "✅ 生成 %s（%.1f MB），耗时 %.1f s\n"
            % (target, os.path.getsize(target) / 1048576, time.time() - start)
        )
        return True

    print("❌ 返回码 %d，未生成 .om。完整日志：%s" % (proc.returncode, log_path))
    # 根因在带错误码的那几行上，它们通常靠前；末尾往往只是一句收尾提示。
    # 错误码行的下一行若是缩进的，通常是与它配套的说明，一并带上。
    lines = log.splitlines()
    coded = []
    for k, line in enumerate(lines):
        if re.search(r"\bE[A-Z]?\d{4,5}\b", line):
            coded.append(line)
            nxt = lines[k + 1] if k + 1 < len(lines) else ""
            if nxt[:1] in (" ", "\t") and nxt.strip():
                coded.append(nxt)
    shown = coded[:10] if coded else [l for l in lines if l.strip()][-12:]
    for line in shown:
        print("   " + line.strip()[:200])
    if not coded:
        print("   （日志里没有错误码行，请打开上面那个文件看全文）")
    print()
    return False


ONNX_OK = describe_onnx(ONNX_PATH) if os.path.exists(ONNX_PATH) else False
if not os.path.exists(ONNX_PATH):
    print("❌ 找不到 %s，请先完成 §5.1 的导出\n" % ONNX_PATH)

INPUT_SHAPE = "--input_shape=%s:1,%d,%d,%d" % (
    INPUT_NAME,
    CHANNELS,
    MODEL_SIDE,
    MODEL_SIDE,
)
OM_PLAIN = "model/yolov13.om"  # v1 用：输入是 FLOAT NCHW
OM_AIPP = "model/yolov13_aipp.om"  # v2 用：输入是 UINT8 NHWC，AIPP 在模型内部
PLAIN_OK = ONNX_OK and run_atc("yolov13", [INPUT_SHAPE])
AIPP_OK = ONNX_OK and run_atc(
    "yolov13_aipp", [INPUT_SHAPE, "--insert_op_conf=" + AIPP_CONF]
)
print(
    "可用模型  :",
    [p for p, ok in ((OM_PLAIN, PLAIN_OK), (OM_AIPP, AIPP_OK)) if ok] or "（未生成）",
)


## 7. 程序实现

**部署链路的第三步。** 《应用开发指南》给出的接口调用次序是：

<img src="images/07.07_infer_flow.png" alt="接口调用流程图" width="620">

图中蓝色是必选步骤，绿色是可选步骤。本实验的程序与它逐项对应：**初始化**是 `aclInit` 加 `aclrtSetDevice` 加 `aclrtCreateContext`，**模型加载**是 `aclmdlLoadFromFile`，**模型执行**是 `aclmdlExecute`，**模型卸载**是 `aclmdlUnload`，**去初始化**是 `aclrtDestroyContext` 加 `aclrtResetDevice` 加 `aclFinalize`，次序与初始化相反。**走 AIPP 这条路，这一串里一个接口都不用增加**——AIPP 在 §6 已经编进了 `.om`。

**图中两个绿色的可选步骤，本实验都不调用。** 「媒体数据处理」是 §2.1 那条路，本实验不需要；「数据后处理」本实验放在主机上自己写。

程序分五段写入 `src_detect/acl_detect.cpp`。两个版本共用这一份代码：从加载模型到取回结果的每一步都相同，**只在预处理这一段有区别**——v1 调 `PreprocessOnHost` 再传 FLOAT，v2 直接传 UINT8。

### 7.1 头文件、常量与错误检查

只有一个头文件：`acl/acl.h`，运行时与模型管理都在里面。**走 AIPP 这条路不需要任何额外的接口**（§2.3）。常量里的 `kModelSide` 是 $640$，图片进入这个程序时已经 letterbox 到这个尺寸（§5.3），因此这里没有缩放要做。


In [ ]:
%%writefile  src_detect/acl_detect.cpp
/**
 * Parallel Computing, Chapter 7, Lab 7: Deploying a Detection Model
 *
 * Two arrangements of one application. Both are handed the same letterboxed
 * 640x640 uint8 RGB image and both produce the same detections. They differ in
 * who turns that image into the float NCHW tensor the network wants -- the
 * conversion to float, the division by 255 and the move to the planar layout:
 *
 *   v1  the host does it, then copies 640*640*3 floats to the device and runs
 *       a model built without AIPP
 *   v2  the host copies the 640*640*3 bytes as they are and runs a model built
 *       with static AIPP, which does the same three steps inside the model
 *
 * So v1 sends four bytes per channel and v2 one, and v1 spends host time that
 * v2 does not. The model is YOLOv13: one 640x640 image in, one (1, 84, 8400)
 * tensor out. The post-processing is identical in both.
 *
 * Usage: acl_detect <mode> <om> <raw> <out> [repeat]
 *   v1 takes the model without AIPP, v2 the one with it.
 */
#include <algorithm>  // std::sort, std::max, std::min
#include <cstdint>    // int64_t, uint8_t, uint32_t, uint64_t
#include <cstdio>     // std::printf, std::fopen
#include <cstdlib>    // std::atoi
#include <cstring>    // std::strcmp
#include <ctime>      // clock_gettime, timespec
#include <vector>     // std::vector

#include "acl/acl.h"  // runtime and model management

namespace {

constexpr int32_t kDeviceId = 0;
constexpr int kWarmupRuns = 2;
constexpr int kDefaultRepeat = 10;
// The image reaches this program already letterboxed to the model's input
// size, so there is no resizing left to do on either path.
constexpr int64_t kModelSide = 640;
constexpr int64_t kChannels = 3;
constexpr int64_t kNumAnchors = 8400;
constexpr int64_t kNumClasses = 80;
constexpr float kConfThresh = 0.25f;
constexpr float kIouThresh = 0.45f;
constexpr int kMaxReported = 16;

// The 80 class names of the COCO dataset, in the order the model scores them.
const char* const kCocoLabels[kNumClasses] = {
    "person",        "bicycle",      "car",
    "motorcycle",    "airplane",     "bus",
    "train",         "truck",        "boat",
    "traffic light", "fire hydrant", "stop sign",
    "parking meter", "bench",        "bird",
    "cat",           "dog",          "horse",
    "sheep",         "cow",          "elephant",
    "bear",          "zebra",        "giraffe",
    "backpack",      "umbrella",     "handbag",
    "tie",           "suitcase",     "frisbee",
    "skis",          "snowboard",    "sports ball",
    "kite",          "baseball bat", "baseball glove",
    "skateboard",    "surfboard",    "tennis racket",
    "bottle",        "wine glass",   "cup",
    "fork",          "knife",        "spoon",
    "bowl",          "banana",       "apple",
    "sandwich",      "orange",       "broccoli",
    "carrot",        "hot dog",      "pizza",
    "donut",         "cake",         "chair",
    "couch",         "potted plant", "bed",
    "dining table",  "toilet",       "tv",
    "laptop",        "mouse",        "remote",
    "keyboard",      "cell phone",   "microwave",
    "oven",          "toaster",      "sink",
    "refrigerator",  "book",         "clock",
    "vase",          "scissors",     "teddy bear",
    "hair drier",    "toothbrush"};

}  // namespace

// One detected object, in the coordinate frame of the model input.
struct Detection {
  float x1;
  float y1;
  float x2;
  float y2;
  float conf;
  int class_id;
};

// Checks the return code of an acl API. On failure it prints the API name,
// the return code and the error message, then returns immediately.
#define ACL_CHECK(expr)                                                     \
  do {                                                                      \
    const int acl_ret = static_cast<int>(expr);                             \
    if (acl_ret != ACL_SUCCESS) {                                           \
      const char* err_msg = aclGetRecentErrMsg();                           \
      std::fprintf(stderr, "[ERR] api=%s code=%d msg=%s\n", #expr, acl_ret, \
                   (err_msg == nullptr) ? "(no message)" : err_msg);        \
      return acl_ret;                                                       \
    }                                                                       \
  } while (0)


### 7.2 主机侧工具：计时、读写文件、预处理与后处理

`PreprocessOnHost` 是 v1 花时间的地方，也**正是 AIPP 为 v2 做的预处理**：把 $640\times640\times3$ 个字节逐个变成 FLOAT、除以 255，并从交错排布（HWC）换成平面排布（CHW）。三重循环，$1228800$ 个元素，**这就是本实验要测的那一段主机计算**。

后处理有三个函数，合起来把模型的原始输出变成一张目标清单：

- **`ParseOutput`** 读那个 $(1, 84, 8400)$ 的张量。8400 是候选框的个数，84 是每个候选框的 $4 + 80$ 个数：前四个是框的中心与宽高，后八十个是每一类的得分。**8400 这一维是变化最快的**，因此同一个候选框的相邻两个数在内存里相隔 8400 个元素；读法一旦弄错，解出来的框全是噪声。
- **`Nms`** 去掉重复框。同一个物体常常被好几个候选框同时框住，NMS 的规则是：**按置信度从高到低，留下最高的那个，把与它重叠超过阈值的同类框全部丢掉。**
- **`PrintDetections`** 把留下来的框按记录行打印出来。打印之前先由 `WriteLabel` 把类别名里的空格换成下划线。

**这三个函数就是后处理的实际代价**：8400 × 80 次比较加一次排序。分类模型的后处理只需要取一个最大值，检测模型则重得多。§10 会把它在三段里的占比测出来。


In [ ]:
%%writefile -a src_detect/acl_detect.cpp
// Returns a monotonic timestamp in milliseconds, for the host-side clock.
double GetTimeMs() {
  timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return static_cast<double>(ts.tv_sec) * 1000.0 +
         static_cast<double>(ts.tv_nsec) / 1.0e6;
}

// Reads a whole file into a host buffer.
int ReadFile(const char* path, std::vector<unsigned char>* content) {
  FILE* handle = std::fopen(path, "rb");
  if (handle == nullptr) {
    std::fprintf(stderr, "[ERR] api=fopen code=- msg=cannot open %s\n", path);
    return ACL_ERROR_INVALID_PARAM;
  }
  std::fseek(handle, 0, SEEK_END);
  const long size = std::ftell(handle);
  std::fseek(handle, 0, SEEK_SET);
  content->assign(static_cast<size_t>(size), 0);
  const size_t got =
      std::fread(content->data(), 1, static_cast<size_t>(size), handle);
  std::fclose(handle);
  return (got == static_cast<size_t>(size)) ? ACL_SUCCESS
                                            : ACL_ERROR_INVALID_PARAM;
}

// Writes a host buffer out to a file.
int WriteFile(const char* path, const void* data, size_t size) {
  FILE* handle = std::fopen(path, "wb");
  if (handle == nullptr) {
    std::fprintf(stderr, "[ERR] api=fopen code=- msg=cannot write %s\n", path);
    return ACL_ERROR_INVALID_PARAM;
  }
  const size_t put = std::fwrite(data, 1, size, handle);
  std::fclose(handle);
  return (put == size) ? ACL_SUCCESS : ACL_ERROR_INVALID_PARAM;
}

// The three steps between a letterboxed uint8 image and the tensor the network
// wants: widen each byte to a float, divide by 255, and move from the
// interleaved layout an image has (HWC) to the planar one (CHW).
//
// This function IS v1's preprocessing, and it is exactly what AIPP does for v2
// inside the model. It reads 640*640*3 bytes and writes as many floats.
void PreprocessOnHost(const unsigned char* source, float* target) {
  const int64_t plane = kModelSide * kModelSide;
  for (int64_t c = 0; c < kChannels; ++c) {
    for (int64_t y = 0; y < kModelSide; ++y) {
      const unsigned char* row = source + y * kModelSide * kChannels;
      float* out = target + c * plane + y * kModelSide;
      for (int64_t x = 0; x < kModelSide; ++x) {
        out[x] = static_cast<float>(row[x * kChannels + c]) / 255.0f;
      }
    }
  }
}

// Intersection over union of two boxes, the measure NMS decides overlap by.
float ComputeIoU(const Detection& a, const Detection& b) {
  const float width =
      std::max(0.0f, std::min(a.x2, b.x2) - std::max(a.x1, b.x1));
  const float height =
      std::max(0.0f, std::min(a.y2, b.y2) - std::max(a.y1, b.y1));
  const float overlap = width * height;
  const float area_a = (a.x2 - a.x1) * (a.y2 - a.y1);
  const float area_b = (b.x2 - b.x1) * (b.y2 - b.y1);
  return overlap / (area_a + area_b - overlap + 1e-6f);
}

// Turns the raw output tensor into a list of boxes above the confidence
// threshold. The tensor is (1, 84, 8400): the 8400 anchors are the fastest
// moving axis, so the four box numbers and the eighty class scores of one
// anchor sit 8400 elements apart from each other.
std::vector<Detection> ParseOutput(const float* data, int64_t anchors) {
  std::vector<Detection> found;
  for (int64_t i = 0; i < anchors; ++i) {
    const float* anchor = data + i;
    float best_score = 0.0f;
    int best_class = 0;
    for (int64_t c = 0; c < kNumClasses; ++c) {
      const float score = anchor[(4 + c) * anchors];
      if (score > best_score) {
        best_score = score;
        best_class = static_cast<int>(c);
      }
    }
    if (best_score < kConfThresh) {
      continue;
    }
    const float cx = anchor[0];
    const float cy = anchor[anchors];
    const float width = anchor[anchors * 2];
    const float height = anchor[anchors * 3];
    Detection det;
    det.x1 = std::max(0.0f, cx - width * 0.5f);
    det.y1 = std::max(0.0f, cy - height * 0.5f);
    det.x2 = std::min(cx + width * 0.5f, static_cast<float>(kModelSide));
    det.y2 = std::min(cy + height * 0.5f, static_cast<float>(kModelSide));
    det.conf = best_score;
    det.class_id = best_class;
    found.push_back(det);
  }
  return found;
}

// Non-maximum suppression: keep the most confident box of a cluster and drop
// every box of the same class that overlaps it too much.
std::vector<Detection> Nms(std::vector<Detection>* candidates) {
  std::sort(
      candidates->begin(), candidates->end(),
      [](const Detection& a, const Detection& b) { return a.conf > b.conf; });
  std::vector<Detection> kept;
  std::vector<char> dropped(candidates->size(), 0);
  for (size_t i = 0; i < candidates->size(); ++i) {
    if (dropped[i] != 0) {
      continue;
    }
    kept.push_back((*candidates)[i]);
    for (size_t j = i + 1; j < candidates->size(); ++j) {
      if (dropped[j] == 0 &&
          (*candidates)[i].class_id == (*candidates)[j].class_id &&
          ComputeIoU((*candidates)[i], (*candidates)[j]) > kIouThresh) {
        dropped[j] = 1;
      }
    }
  }
  return kept;
}

// Turns a class name into one token: the records below are parsed by
// splitting on whitespace, so a value must not contain a space. Fifteen of
// the eighty COCO names do ("traffic light", "cell phone", ...).
void WriteLabel(int class_id, char* out, size_t size) {
  const char* name = kCocoLabels[class_id];
  size_t i = 0;
  for (; name[i] != '\0' && i + 1 < size; ++i) {
    out[i] = (name[i] == ' ') ? '_' : name[i];
  }
  out[i] = '\0';
}

// Prints one record line per detected object, most confident first.
void PrintDetections(const char* version, const std::vector<Detection>& dets) {
  const size_t shown = (dets.size() < static_cast<size_t>(kMaxReported))
                           ? dets.size()
                           : static_cast<size_t>(kMaxReported);
  char label[32];
  for (size_t i = 0; i < shown; ++i) {
    WriteLabel(dets[i].class_id, label, sizeof(label));
    std::printf(
        "[DET] version=%s index=%zu label=%s conf=%.4f x1=%d y1=%d x2=%d "
        "y2=%d\n",
        version, i, label, static_cast<double>(dets[i].conf),
        static_cast<int>(dets[i].x1), static_cast<int>(dets[i].y1),
        static_cast<int>(dets[i].x2), static_cast<int>(dets[i].y2));
  }
}


### 7.3 一次部署要准备的资源

`Runner` 把一次部署要在设备上持有的东西集中在一处：一份模型实例与它的描述、一对输入输出 Dataset。**两个版本持有的东西完全一样**，差别只在加载哪一个 `.om`，以及往输入缓冲区里写什么。

其中输入输出 Dataset 的准备是一段固定的次序，《应用开发指南》给出的流程是：

<img src="images/07.07_dataset.png" alt="模型执行的输入/输出数据结构的准备流程" width="520">

`CreateDataset` 实现的就是这条流程：`aclmdlCreateDataset` 建容器，`aclmdlGetNumInputs` / `aclmdlGetNumOutputs` 取个数，然后按个数循环——`aclmdlGetInputSizeByIndex` / `aclmdlGetOutputSizeByIndex` 问尺寸、`aclrtMalloc` 申请、`aclCreateDataBuffer` 包装、`aclmdlAddDatasetBuffer` 挂进容器。

图中绿色那一格「获取输入/输出的名称」是可选步骤，**本实验不需要**：模型只有一个输入和一个输出，按索引 $0$ 定位没有歧义。多输入的模型才要按名称对齐，否则各输入的次序由模型转换时决定，按索引取容易取错。

两处值得注意：

- **尺寸一律向模型查询，不由程序假定**——`aclmdlGetInputSizeByIndex` 返回的字节数是唯一可信的来源；
- **拿到之后立刻核对**：带 AIPP 的模型每通道一个字节，不带的四个字节，`CreateRunner` 用 `aipp` 这个标志算出应有的值并逐字节比对。


In [ ]:
%%writefile -a src_detect/acl_detect.cpp
// Everything one deployment holds on the device: the model, its description,
// and the input and output datasets. Both arrangements hold exactly this; they
// differ only in which model file they load and what they write into the input
// buffer.
struct Runner {
  uint32_t model_id;
  aclmdlDesc* desc;
  size_t input_bytes;
  size_t output_bytes;
  aclmdlDataset* input;
  aclmdlDataset* output;
  std::vector<float> host_output;
};

// Builds one dataset with one device buffer per model input or output.
int CreateDataset(aclmdlDesc* desc, bool is_input, aclmdlDataset** dataset) {
  *dataset = aclmdlCreateDataset();
  if (*dataset == nullptr) {
    return ACL_ERROR_INVALID_PARAM;
  }
  const size_t count =
      is_input ? aclmdlGetNumInputs(desc) : aclmdlGetNumOutputs(desc);
  for (size_t i = 0; i < count; ++i) {
    const size_t bytes = is_input ? aclmdlGetInputSizeByIndex(desc, i)
                                  : aclmdlGetOutputSizeByIndex(desc, i);
    void* device = nullptr;
    ACL_CHECK(aclrtMalloc(&device, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMemset(device, bytes, 0, bytes));
    aclDataBuffer* buffer = aclCreateDataBuffer(device, bytes);
    if (buffer == nullptr) {
      // No buffer has taken ownership of this block yet, so nothing else will
      // ever give it back.
      (void)aclrtFree(device);
      return ACL_ERROR_INVALID_PARAM;
    }
    ACL_CHECK(aclmdlAddDatasetBuffer(*dataset, buffer));
  }
  return ACL_SUCCESS;
}

// Releases a dataset: address out first, then the memory, then the buffer.
void DestroyDataset(aclmdlDataset* dataset) {
  if (dataset == nullptr) {
    return;
  }
  for (size_t i = 0; i < aclmdlGetDatasetNumBuffers(dataset); ++i) {
    aclDataBuffer* buffer = aclmdlGetDatasetBuffer(dataset, i);
    (void)aclrtFree(aclGetDataBufferAddr(buffer));
    (void)aclDestroyDataBuffer(buffer);
  }
  (void)aclmdlDestroyDataset(dataset);
}

// Loads the model and sizes every buffer from what the model itself reports.
//
// `aipp` says which of the two models this is, and the only thing it changes
// here is the expected size of the input buffer: with AIPP the model takes one
// byte per channel, without it four. Checking this is what catches the two
// files being swapped by mistake.
int CreateRunner(Runner* r, const char* om_path, bool aipp) {
  ACL_CHECK(aclmdlLoadFromFile(om_path, &r->model_id));
  r->desc = aclmdlCreateDesc();
  ACL_CHECK(aclmdlGetDesc(r->desc, r->model_id));
  r->input_bytes = aclmdlGetInputSizeByIndex(r->desc, 0);
  r->output_bytes = aclmdlGetOutputSizeByIndex(r->desc, 0);

  const size_t element = aipp ? 1 : sizeof(float);
  const size_t wanted =
      static_cast<size_t>(kModelSide * kModelSide * kChannels) * element;
  if (r->input_bytes != wanted) {
    std::fprintf(stderr,
                 "[ERR] api=CreateRunner code=- msg=model input is %zu bytes, "
                 "this path needs %zu; is %s the right model for this mode?\n",
                 r->input_bytes, wanted, om_path);
    return ACL_ERROR_INVALID_PARAM;
  }
  const size_t output_wanted =
      static_cast<size_t>((4 + kNumClasses) * kNumAnchors) * sizeof(float);
  if (r->output_bytes < output_wanted) {
    std::fprintf(stderr,
                 "[ERR] api=CreateRunner code=- msg=model output is %zu bytes, "
                 "smaller than the %zu the post-processing reads\n",
                 r->output_bytes, output_wanted);
    return ACL_ERROR_INVALID_PARAM;
  }

  ACL_CHECK(CreateDataset(r->desc, true, &r->input));
  ACL_CHECK(CreateDataset(r->desc, false, &r->output));
  r->host_output.assign(r->output_bytes / sizeof(float), 0.0f);
  return ACL_SUCCESS;
}

void DestroyRunner(Runner* r) {
  DestroyDataset(r->output);
  DestroyDataset(r->input);
  r->output = nullptr;
  r->input = nullptr;
  // The description is created right after the load succeeds, so a null one
  // means there is no model to unload -- which is the state CreateRunner
  // leaves behind when it fails on its very first step.
  if (r->desc != nullptr) {
    (void)aclmdlUnload(r->model_id);
    (void)aclmdlDestroyDesc(r->desc);
    r->desc = nullptr;
  }
}


### 7.4 两条路：v1 与 v2

两个函数的三段计时方式完全一致，因此可以逐段对照：

- **第一段（预处理）**：**两版的起点都是主机内存里那份 letterbox 好的 UINT8 图像，终点都是设备上模型输入缓冲区里已经就绪的数据。** v1 是 `PreprocessOnHost` 加一次 4915200 字节的传输；v2 只有一次 1228800 字节的传输；
- **第二段（推理）**：两版都是 `aclmdlExecute`，但**模型不同**——v2 的模型里多了 AIPP 这一段，**v1 省下的主机时间，有一部分会在这里重新出现**；
- **第三段（后处理）**：两版都是取回原始输出加一次 NMS，**代码相同、输入规模也相同**（都是 $84\times8400$ 个 FLOAT）。唯一可能不同的是过阈的候选框个数——它取决于置信度，而两版的置信度有微小差别，NMS 的比较次数因此可以略有出入。

预期是：第一段 v2 明显更快，第二段 v2 略慢，第三段两版相当。**净收益是三段之和，也就是端到端**。

每一版都先跑 `kWarmupRuns` 次不计时的预热。首次执行要做算子编译与内存分配，把它算进平均值会掩盖稳定之后的真实耗时。

`[VER]` 记录里多带了一个 `h2d_bytes`，把这一版实际送出去的字节数一并输出出来。

In [ ]:
%%writefile -a src_detect/acl_detect.cpp
// Copies the raw output tensor back to the host.
int FetchOutput(Runner* r) {
  aclDataBuffer* buffer = aclmdlGetDatasetBuffer(r->output, 0);
  ACL_CHECK(aclrtMemcpy(r->host_output.data(), r->output_bytes,
                        aclGetDataBufferAddr(buffer), r->output_bytes,
                        ACL_MEMCPY_DEVICE_TO_HOST));
  return ACL_SUCCESS;
}

// Turns the raw output into a list of boxes.
std::vector<Detection> PostProcess(const Runner& r) {
  std::vector<Detection> candidates =
      ParseOutput(r.host_output.data(), kNumAnchors);
  return Nms(&candidates);
}

// Prints one record with the three segments and the end-to-end time, plus the
// number of bytes this arrangement sent to the device.
void ReportVersion(const char* label, double pre_ms, double infer_ms,
                   double post_ms, size_t h2d_bytes, size_t objects) {
  const double e2e_ms = pre_ms + infer_ms + post_ms;
  std::printf(
      "[VER] version=%s pre_ms=%.4f infer_ms=%.4f post_ms=%.4f e2e_ms=%.4f "
      "fps=%.2f h2d_bytes=%zu objects=%zu\n",
      label, pre_ms, infer_ms, post_ms, e2e_ms, 1000.0 / e2e_ms, h2d_bytes,
      objects);
}

// v1: the host widens, scales and transposes the image, then sends the float
// tensor to a model built without AIPP.
int RunVersion1(Runner* r, const std::vector<unsigned char>& source, int repeat,
                const char* out_path) {
  const size_t elems = static_cast<size_t>(kModelSide * kModelSide * kChannels);
  const size_t in_bytes = elems * sizeof(float);
  std::vector<float> staging(elems);
  aclDataBuffer* in_buffer = aclmdlGetDatasetBuffer(r->input, 0);

  double pre_total = 0.0;
  double infer_total = 0.0;
  double post_total = 0.0;
  std::vector<Detection> dets;
  for (int run = 0; run < kWarmupRuns + repeat; ++run) {
    const double t0 = GetTimeMs();
    PreprocessOnHost(source.data(), staging.data());
    ACL_CHECK(aclrtMemcpy(aclGetDataBufferAddr(in_buffer), in_bytes,
                          staging.data(), in_bytes, ACL_MEMCPY_HOST_TO_DEVICE));
    const double t1 = GetTimeMs();
    ACL_CHECK(aclmdlExecute(r->model_id, r->input, r->output));
    const double t2 = GetTimeMs();
    ACL_CHECK(FetchOutput(r));
    dets = PostProcess(*r);
    const double t3 = GetTimeMs();
    if (run >= kWarmupRuns) {
      pre_total += t1 - t0;
      infer_total += t2 - t1;
      post_total += t3 - t2;
    }
  }
  ReportVersion("v1", pre_total / repeat, infer_total / repeat,
                post_total / repeat, in_bytes, dets.size());
  PrintDetections("v1", dets);
  if (out_path != nullptr) {
    ACL_CHECK(WriteFile(out_path, r->host_output.data(), r->output_bytes));
  }
  return ACL_SUCCESS;
}

// v2: the host sends the letterboxed image as it is, one byte per channel, to
// a model built with static AIPP. The preprocessing segment is then a single
// copy: the three steps v1 does on the host are inside the model.
int RunVersion2(Runner* r, const std::vector<unsigned char>& source, int repeat,
                const char* out_path) {
  const size_t in_bytes = static_cast<size_t>(kModelSide * kModelSide * kChannels);
  aclDataBuffer* in_buffer = aclmdlGetDatasetBuffer(r->input, 0);

  double pre_total = 0.0;
  double infer_total = 0.0;
  double post_total = 0.0;
  std::vector<Detection> dets;
  for (int run = 0; run < kWarmupRuns + repeat; ++run) {
    const double t0 = GetTimeMs();
    ACL_CHECK(aclrtMemcpy(aclGetDataBufferAddr(in_buffer), in_bytes,
                          source.data(), in_bytes, ACL_MEMCPY_HOST_TO_DEVICE));
    const double t1 = GetTimeMs();
    ACL_CHECK(aclmdlExecute(r->model_id, r->input, r->output));
    const double t2 = GetTimeMs();
    ACL_CHECK(FetchOutput(r));
    dets = PostProcess(*r);
    const double t3 = GetTimeMs();
    if (run >= kWarmupRuns) {
      pre_total += t1 - t0;
      infer_total += t2 - t1;
      post_total += t3 - t2;
    }
  }
  ReportVersion("v2", pre_total / repeat, infer_total / repeat,
                post_total / repeat, in_bytes, dets.size());
  PrintDetections("v2", dets);
  if (out_path != nullptr) {
    ACL_CHECK(WriteFile(out_path, r->host_output.data(), r->output_bytes));
  }
  return ACL_SUCCESS;
}


### 7.5 主程序

`SetUp` 与 `TearDown` 是成对的：申请资源的次序与释放的次序相反。**`SetUp` 自身也可能在中途失败**，此时前面几步已经申请到的资源仍然要还回去，因此它的失败路径同样经过 `TearDown`。

`Dispatch` 按模式选路，**只有一处分支**：`v2` 走 AIPP，其余走主机侧。这一个布尔值决定两件事——`CreateRunner` 按几个字节校验模型输入，以及调用哪一个 `RunVersion`。

读入图片之后立刻核对它的字节数。两条路都按 `kModelSide` 这个编译期常量读这块内存，尺寸对不上就是越界读。

释放的写法值得注意：`DestroyRunner` **在成功与失败两条路上都会执行**。`CreateRunner` 中途失败时，前面已经申请到的 Dataset 与设备内存仍然要释放；把释放写在分支外面，这一点就由代码结构本身保证。

In [ ]:
%%writefile -a src_detect/acl_detect.cpp
// Acquires the runtime resources both arrangements need.
int SetUp(aclrtContext* context) {
  ACL_CHECK(aclInit(nullptr));
  ACL_CHECK(aclrtSetDevice(kDeviceId));
  ACL_CHECK(aclrtCreateContext(context, kDeviceId));
  ACL_CHECK(aclrtSetCurrentContext(*context));
  std::printf("[ENV] soc=%s\n", aclrtGetSocName());
  return ACL_SUCCESS;
}

void TearDown(aclrtContext context) {
  if (context != nullptr) {
    (void)aclrtDestroyContext(context);
  }
  (void)aclrtResetDevice(kDeviceId);
  (void)aclFinalize();
}

// Reads the arguments, builds the runner the chosen version needs, runs it,
// and releases everything on the way out -- including the failing paths.
int Dispatch(int argc, char** argv) {
  const char* mode = (argc > 1) ? argv[1] : "v1";
  const char* om_path = (argc > 2) ? argv[2] : "model/yolov13.om";
  const char* raw_path = (argc > 3) ? argv[3] : "data/bus_letterbox.bin";
  const char* out_path = (argc > 4) ? argv[4] : "data/raw_v1.bin";
  const int repeat = (argc > 5) ? std::atoi(argv[5]) : kDefaultRepeat;
  const bool aipp = (std::strcmp(mode, "v2") == 0);

  aclrtContext context = nullptr;
  const int ready = SetUp(&context);
  if (ready != ACL_SUCCESS) {
    // SetUp can fail part of the way through; whatever it did acquire still
    // has to be given back.
    TearDown(context);
    return ready;
  }
  std::vector<unsigned char> source;
  int status = ReadFile(raw_path, &source);
  if (status != ACL_SUCCESS) {
    TearDown(context);
    return status;
  }
  // Both paths read this buffer at a size fixed at compile time, so check the
  // file really is that large. Same principle as the check in CreateRunner
  // against the size the model reports: whatever a size comes from, compare
  // against it before using it.
  const size_t expected =
      static_cast<size_t>(kModelSide * kModelSide * kChannels);
  if (source.size() != expected) {
    std::fprintf(stderr,
                 "[ERR] api=Dispatch code=- msg=%s is %zu bytes, expected %zu\n",
                 raw_path, source.size(), expected);
    TearDown(context);
    return ACL_ERROR_INVALID_PARAM;
  }

  Runner runner = {};
  status = CreateRunner(&runner, om_path, aipp);
  if (status == ACL_SUCCESS) {
    status = aipp ? RunVersion2(&runner, source, repeat, out_path)
                  : RunVersion1(&runner, source, repeat, out_path);
  }
  DestroyRunner(&runner);

  TearDown(context);
  std::printf("[RESULT] %s\n", (status == ACL_SUCCESS) ? "PASS" : "FAIL");
  return status;
}

int main(int argc, char** argv) {
  return (Dispatch(argc, argv) == ACL_SUCCESS) ? 0 : 1;
}


## 8. 编译与运行

**部署链路的第四步。** 程序只用到 `acl/acl.h` 里的接口，因此编译只要带上 §4 探测出的头文件搜索路径与那两个库。

**AIPP 在这里是看不见的**：它已经在 §6 编译进了 `.om`，运行期不需要任何额外的接口、库或代码。这是 AIPP 的一个重要性质——**把预处理交给它，应用程序反而变得更简单**。


In [ ]:
import subprocess

cmd = (
    ["g++", "src_detect/acl_detect.cpp", "-std=c++17", "-O2", "-Wall", "-Wextra"]
    + ACL_INCDIRS
    + ACL_LIBDIRS
    + ACL_LIBS
    + ["-o", "src_detect/acl_detect"]
)
print(" ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print((proc.stdout + proc.stderr).strip() or "✅ 编译通过")


两个版本各跑一次，**读的是同一份 letterbox 结果**（§5.3），只是模型不同（§6）。每一版内部重复 `REPEAT` 次取平均，另有两次不计时的预热（§7.4）。


In [ ]:
import subprocess

REPEAT = 10
EXE = "./src_detect/acl_detect"
SRC = LETTERBOX_PATH  # §5.3 的产物，两版共用
out = {}

# 两次运行读同一份输入，差别只在模式与模型
runs = []
if PLAIN_OK:
    runs.append(("v1", [EXE, "v1", OM_PLAIN, SRC, "data/raw_v1.bin", str(REPEAT)]))
if AIPP_OK:
    runs.append(("v2", [EXE, "v2", OM_AIPP, SRC, "data/raw_v2.bin", str(REPEAT)]))

def diagnose(text):
    "把已知的几类失败归纳为一句结论与一条处理办法。"
    if "is the right model for this mode" in text:
        print(
            "\n    成因：模型与模式对不上。v1 要不带 AIPP 的模型（输入是 FLOAT），"
            "\n          v2 要带 AIPP 的模型（输入是 UINT8）。请核对 §6 生成的两个文件。"
        )
    elif "aclmdlLoadFromFile" in text:
        print("\n    成因：模型加载失败，请先确认 §6 的两个 .om 都已生成。")
    elif "cannot open" in text or "expected" in text:
        print("\n    成因：输入文件缺失或大小不对，请先运行 §5.3。")


# 每个版本都设上限：超时的那一个被终止并记下来，另一个照常跑完。
RUN_TIMEOUT_S = 300

for name, cmd in runs:
    print(f"=== {name:4s} ", end="", flush=True)
    try:
        proc = subprocess.run(
            cmd, capture_output=True, text=True, timeout=RUN_TIMEOUT_S
        )
    except subprocess.TimeoutExpired as expired:
        captured = expired.stdout or ""
        if isinstance(captured, bytes):
            captured = captured.decode("utf-8", "replace")
        out[name] = captured
        print(f"⏱️  超过 {RUN_TIMEOUT_S} s 未结束，已终止")
        print("    " + " ".join(cmd))
        continue
    out[name] = proc.stdout
    # 直接找 [RESULT]，不取最后一行：算子库会在 stdout 上留下不带换行的片段，
    # 最后一行未必以 [RESULT] 开头（§9 对此有说明）。
    mark = proc.stdout.rfind("[RESULT]")
    tail = proc.stdout[mark:].strip() if mark >= 0 else "(没有 [RESULT] 行)"
    print(f"退出码 {proc.returncode}  {tail}")
    if proc.returncode != 0:
        print(proc.stderr.strip()[:600])
        diagnose(proc.stderr)


## 9. 解析输出

记录有两种：`[VER]` 报一个版本的三段耗时与它送出去的字节数，`[DET]` 报一个检出的目标。两种共用一个解析函数。


In [ ]:
NUMERIC = {
    "pre_ms",
    "infer_ms",
    "post_ms",
    "e2e_ms",
    "fps",
    "h2d_bytes",
    "objects",
    "index",
    "conf",
    "x1",
    "y1",
    "x2",
    "y2",
}


def parse(text, tag):
    "把带某个标签的所有记录都解析成字典。数值字段取到 - 时不收这个键。"
    rows = []
    # 按标签切分，而不是按行首匹配：记录未必从行首开始（见本节说明）。
    for chunk in text.split(tag + " ")[1:]:
        item = {}
        for token in chunk.split("\n", 1)[0].split():
            if "=" not in token:
                continue
            key, value = token.split("=", 1)
            if key in NUMERIC:
                if value != "-":
                    item[key] = float(value)
            else:
                item[key] = value
        rows.append(item)
    return rows


records = []
detections = []
for name, text in out.items():
    for row in parse(text, "[VER]"):
        row["run"] = name
        records.append(row)
    detections.extend(parse(text, "[DET]"))

v1 = [r for r in records if r["version"] == "v1"]
v2 = [r for r in records if r["version"] == "v2"]
print(
    "解析到：v1 %d 行，v2 %d 行，检出 %d 个目标"
    % (len(v1), len(v2), len(detections))
)


## 10. 阶段分解：v1 与 v2

三段并排。**要看的不是哪一版更快，而是差别在哪一段、为什么。**

这个对照是受控的：两版读同一份 letterbox 结果（§5.3），检出同样的目标（下面会核对），程序是同一份，只有预处理的地方不同。因此预期是明确的：

- **预处理段**：v2 应当明显更快——主机不再逐像素算，送出去的数据也少了四分之三；
- **推理段**：v2 应当略慢——AIPP 是模型的一部分，它做的事要占设备的时间（可能存在计时波动）；
- **后处理段**：两版应当相当——代码与输入规模一样，偏差主要来自过阈候选框个数的小幅不同与测量本身的波动。

**净收益是端到端，不是预处理段。** 下面依次打印三段耗时与占比、两版各自送出去的字节数，以及上面三条的核对结果。

In [ ]:
if not v1 or not v2:
    print("v1 或 v2 没有数据，请先确认 §8 已运行成功。")
else:
    a, b = v1[0], v2[0]
    print(
        "%-18s %12s %12s %12s %12s %14s"
        % ("版本", "预处理 (ms)", "推理 (ms)", "后处理 (ms)", "端到端 (ms)", "H2D (字节)")
    )
    print("-" * 86)
    for label, row in (("v1 主机侧预处理", a), ("v2 AIPP 预处理", b)):
        print(
            "%-19s %12.4f %12.4f %12.4f %12.4f %14d"
            % (
                label,
                row["pre_ms"],
                row["infer_ms"],
                row["post_ms"],
                row["e2e_ms"],
                int(row["h2d_bytes"]),
            )
        )
    print()
    for label, row in (("v1", a), ("v2", b)):
        total = row["e2e_ms"]
        print(
            "%s 三段占比：预处理 %.1f%%，推理 %.1f%%，后处理 %.1f%%"
            % (
                label,
                row["pre_ms"] / total * 100,
                row["infer_ms"] / total * 100,
                row["post_ms"] / total * 100,
            )
        )

    print()
    print("逐段对照（v2 相对 v1，比值小于 1 表示 v2 更快）：")
    for name, key in (("预处理", "pre_ms"), ("推理", "infer_ms"),
                      ("后处理", "post_ms"), ("端到端", "e2e_ms")):
        print(
            "  %-6s %8.4f → %8.4f ms   变化 %+8.4f ms   v2/v1 = %.3f"
            % (name, a[key], b[key], b[key] - a[key],
               b[key] / max(a[key], 1e-9))
        )
    print(
        "  H2D    %8d → %8d 字节   v2 是 v1 的 %.2f"
        % (int(a["h2d_bytes"]), int(b["h2d_bytes"]),
           b["h2d_bytes"] / max(a["h2d_bytes"], 1e-9))
    )

    print()
    print("核对三条预期：")
    pre_gain = a["pre_ms"] - b["pre_ms"]
    infer_cost = b["infer_ms"] - a["infer_ms"]
    post_gap = abs(b["post_ms"] - a["post_ms"]) / max(a["post_ms"], 1e-9)
    print(
        "  ① 预处理段 v2 更快 —— %s（省下 %.4f ms）"
        % ("是" if pre_gain > 0 else "否", pre_gain)
    )
    print(
        "  ② 推理段 v2 更慢 —— %s（多出 %.4f ms，这是 AIPP 在设备上的代价）"
        % ("是" if infer_cost > 0 else "否", infer_cost)
    )
    # 两版这一段的代码相同，输入规模也相同；但过阈的候选框个数取决于置信度，
    # 而两版的置信度有微小差别，NMS 的比较次数因此可以略有不同。加上毫秒级
    # 测量本身的波动，十个百分点以内都属正常。
    print(
        "  ③ 后处理段两版相当 —— %s（相对偏差 %.1f%%）"
        % ("是" if post_gap <= 0.10 else "否，偏差偏大，前两条的结论要谨慎看待",
           post_gap * 100)
    )
    print(
        "\n净收益 = 预处理省下的 - 推理多出的 = %.4f - %.4f = %+.4f ms，"
        "与端到端的变化 %+.4f ms 应当一致（差值来自后处理段的波动）。"
        % (pre_gain, infer_cost, pre_gain - infer_cost, a["e2e_ms"] - b["e2e_ms"])
    )


**判据要留出容差。** 主机做的是 IEEE 单精度的除法与拷贝，AIPP 在设备上有它自己的实现，**两者不是同一套算术**。同一张图经过两条路，得到的模型输入张量在数值上就会有微小差别，这个差别再经过整个网络放大，最后表现为：置信度的末几位不同，框的浮点坐标也略有出入。

**因此只要求个数与类别序列完全一致，框坐标允许两个像素以内的偏差。** 

另打印两个观察量：**框坐标的最大偏差与置信度的最大差值**，以及 §8 写出的两份原始输出张量（`data/raw_v1.bin` 与 `data/raw_v2.bin`，各 $84\times8400$ 个 FLOAT）的**最大绝对差与相对差**。

**相对差在 $10^{-3}$ 这个量级上是正常的**，它反映的是两套算术的差异，不说明配置有错。**判断 AIPP 配置是否正确，看的是检出结果本身**——类别、个数、框的位置与置信度的排序。若这些明显不对，才回头核对配置：最常见的是 `var_reci_chn_*` 写错，或者 `rbuv_swap_switch` 与 OpenCV 的通道转换重复了一次。

In [ ]:
def boxes_of(version):
    "某一版检出的目标，按置信度从高到低。"
    rows = [d for d in detections if d["version"] == version]
    return sorted(rows, key=lambda d: d["index"])


def to_original(box):
    "把模型坐标系里的框映射回原图：减去补边，再除以缩放比。"
    corners = (
        (box["x1"], PAD_X),
        (box["y1"], PAD_Y),
        (box["x2"], PAD_X),
        (box["y2"], PAD_Y),
    )
    return tuple(int(round((v - pad) / SCALE)) for v, pad in corners)


a, b = boxes_of("v1"), boxes_of("v2")
for label, rows in (("v1 主机侧预处理", a), ("v2 AIPP 预处理", b)):
    print("=" * 72)
    print("%s：检出 %d 个目标" % (label, len(rows)))
    print("=" * 72)
    for row in rows:
        print(
            "  [%d] %-14s 置信度 %.4f   模型坐标 [%d,%d,%d,%d]   原图坐标 %s"
            % (
                int(row["index"]),
                row["label"],
                row["conf"],
                int(row["x1"]),
                int(row["y1"]),
                int(row["x2"]),
                int(row["y2"]),
                to_original(row),
            )
        )
    print()

kinds = sorted({row["label"] for row in a})
print(
    "判据 1：检出的类别包含 bus 与 person —— %s（实测 %s）"
    % ("PASS" if {"bus", "person"} <= set(kinds) else "FAIL", kinds)
)
# 框坐标允许的偏差。两条路的算术不同（主机的 IEEE 浮点除法 vs AIPP 在设备上的
# 实现），框的浮点坐标会有微小差别，取整时可能落到相邻的整数上。
BOX_TOL = 2  # 模型坐标（640 尺度）上的像素数

labels_a = [r["label"] for r in a]
labels_b = [r["label"] for r in b]
lined_up = len(a) == len(b) and labels_a == labels_b and bool(a)
if not lined_up:
    same = False
    print(
        "判据 2：目标个数与类别序列相同 —— FAIL"
        "（v1 %d 个 %s；v2 %d 个 %s）" % (len(a), labels_a, len(b), labels_b)
    )
else:
    box_gap = max(
        max(abs(int(x[k]) - int(y[k])) for k in ("x1", "y1", "x2", "y2"))
        for x, y in zip(a, b)
    )
    conf_gap = max(abs(x["conf"] - y["conf"]) for x, y in zip(a, b))
    same = box_gap <= BOX_TOL
    print(
        "判据 2：个数与类别序列相同，且框坐标偏差不超过 %d 像素 —— %s"
        % (BOX_TOL, "PASS" if same else "FAIL")
    )
    print(
        "        实测 %d 个目标，类别序列一致；框坐标最大偏差 %d 像素，"
        "置信度最大差值 %.4f" % (len(a), box_gap, conf_gap)
    )

# 两版的原始输出张量已经写在磁盘上，逐元素比一次，看差别落在哪个量级
RAW = ("data/raw_v1.bin", "data/raw_v2.bin")
if all(os.path.exists(path) for path in RAW):
    x, y = (np.fromfile(path, dtype=np.float32) for path in RAW)
    if x.size == y.size and x.size:
        scale = max(np.abs(x).max(), 1e-9)
        print(
            "观察量：原始输出张量 %d 个元素，最大绝对差 %.3e，"
            "相对于该张量的最大值（%.3f）为 %.3e"
            % (x.size, np.abs(x - y).max(), scale, np.abs(x - y).max() / scale)
        )
    else:
        print("两份原始输出的元素个数不同，请确认 §8 的两次运行都已完成。")
if not same and a and b:
    print(
        "  判据 2 不通过时，请把上面两张清单与 §11 的两张图一并核对："
        "偏差若只有一两个像素，多数是取整落到了相邻整数上；"
        "偏差明显更大，才说明 AIPP 的配置与主机侧的三步转换不是同一件事。"
    )


## 11. 检测结果可视化

逐行核对坐标能确认两条路一致，但看不出检出是否准确。**把框画回原图**即可直接判断。**这是整条部署链路是否正确的最终判据**——从权重到 ONNX、到 `.om`、到程序，中间任何一步出错，框都不会落在应有的位置上。

坐标要走一次逆变换：模型坐标减去补边、再除以缩放比，才落回原图的像素位置——用的就是 §5.3 记下的 `SCALE`、`PAD_X`、`PAD_Y`。


In [ ]:
%matplotlib inline

import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Rectangle

matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # 图内标签统一使用 ASCII
matplotlib.rcParams["axes.unicode_minus"] = False

DETECT_PNG = "data/detections.png"
PALETTE = ["#C7000B", "#295E96", "#C15A00", "#2E7D32", "#6A1B9A", "#00838F"]


def clamp(value, low, high):
    return max(low, min(high, value))


def draw_one(ax, version, caption):
    "把某一版的检出框画到原图上，同一类别用同一种颜色。"
    with Image.open(IMAGE_PATH) as image:
        ax.imshow(image.convert("RGB"))
    rows = boxes_of(version)
    kinds = []
    for row in rows:
        if row["label"] not in kinds:
            kinds.append(row["label"])
    for row in rows:
        x1, y1, x2, y2 = to_original(row)
        x1, x2 = clamp(x1, 0, ORIG_W), clamp(x2, 0, ORIG_W)
        y1, y2 = clamp(y1, 0, ORIG_H), clamp(y2, 0, ORIG_H)
        color = PALETTE[kinds.index(row["label"]) % len(PALETTE)]
        ax.add_patch(
            Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                edgecolor=color,
                linewidth=2.0,
            )
        )
        # 靠右的框把标签右对齐，否则文字会越出图像边界
        at_right = x1 > ORIG_W * 0.7
        ax.text(
            x2 if at_right else x1,
            max(y1 - 4, 14),
            "%s %.2f" % (row["label"], row["conf"]),
            fontsize=8.5,
            color="white",
            ha="right" if at_right else "left",
            va="bottom",
            bbox={"facecolor": color, "edgecolor": "none", "pad": 1.6},
        )
    ax.set_title("%s: %d objects" % (caption, len(rows)), fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


panels = [
    (version, caption)
    for version, caption in (
        ("v1", "v1 host preprocessing"),
        ("v2", "v2 AIPP preprocessing"),
    )
    if boxes_of(version)
]
if not panels:
    print("还没有检出记录，请先确认 §8 已运行成功。")
else:
    fig, axes = plt.subplots(1, len(panels), figsize=(5.0 * len(panels), 6.6), dpi=110)
    axes = [axes] if len(panels) == 1 else list(axes)
    for ax, (version, caption) in zip(axes, panels):
        draw_one(ax, version, caption)
    fig.tight_layout()
    fig.savefig(DETECT_PNG, dpi=110, bbox_inches="tight")
    plt.show()
    print("已保存 :", DETECT_PNG)


## 12. 结果分析

### 🎓 结论

**① 部署一个模型是一条固定的链路，每一步都可能出错，而这些错误未必都会当场报出来。** 权重 → ONNX → `.om` → 程序，四步各有各的约束：导出要选 ATC 支持的算子集版本；转换要给对输入名与输入形状；程序要按模型自己报出来的字节数准备缓冲区。**其中一部分会当场报错**——ATC 给出错误码、缓冲区尺寸不符时程序拒绝执行；**另一部分不会**，例如通道顺序弄反、归一化系数写错，程序照常跑完，只是框不对。因此链路的末端要有一个能直接判断对错的检查，本实验用的是把框画回原图（§11）。

**② 优化之前先测三段的比例。** §10 把端到端拆成预处理、推理、后处理。若预处理占了其中的大部分，优化推理带来的改善十分有限。检测模型的后处理尤其要先量一遍：它要在数千个候选框上比对数十个类别、再排一次序，与分类模型只取一个最大值不是一个量级。应先完成阶段分解，再确定优化的对象。

**③ 用 AIPP 做预处理，在主机侧同时省下两样东西：逐像素的计算，与四分之三的传输量。** 转 FLOAT、除以 255、排布转换这三步，逐像素地做在百万量级的元素上；交给 AIPP 之后主机一步都不做。同时送进设备的由 FLOAT 变为 UINT8，**字节数降到四分之一**。

**④ 代价不在预处理这一段，而在推理那一段。** AIPP 是模型的一部分，它做的事要占设备的时间。**净收益是两者之差，因此只看预处理段会高估 AIPP 的作用，要看端到端。** 

**⑤ AIPP 的工程代价接近于零，这一点与媒体数据处理算子形成对照。** 走 AIPP 这条路，应用程序不增加任何库、接口调用或内存管理——它是在模型转换时由 `--insert_op_conf` 加进去的（§2.3）。走算子那条路则要额外链接库、创建张量描述符、管理 workspace 并显式同步。**参数固定不变的预处理，交给 AIPP 是更省事的一条路。**

**⑥ AIPP 与媒体数据处理算子的分界，是编译期与运行期的分界。** 能在模型转换时定下来的事就写进 `.om`，代价是这些参数被固定在模型里，改一次要重新转换；需要在运行期按每张图变化的事，才留给算子或者动态 AIPP。

## 13. 🔧 动手练习

**1. 量出预处理三步各自的代价。** 把 `PreprocessOnHost` 拆成三个函数分别计时：只做类型转换、只做除以 255、只做排布转换。请回答：① 三者各占这一段的多少？② **排布转换为什么最贵？** 提示：它按 `target[c][y][x] = source[y][x][c]` 写，读一行源数据要跨 3 个字节取一个值，写则是连续的——这与第三章的访存局部性是同一件事。③ 若只把最贵的那一步交给 AIPP、其余两步仍在主机做，能省下多少？

**2. 改一个 AIPP 参数，看它落在哪里。** 把 `var_reci_chn_*` 三行由 $1/255$ 改成 $1/128$，重新转一个 `.om` 并重跑 v2。请回答：① 检出结果怎么变？② 端到端耗时变了吗、为什么？③ **这次改动能不能改在应用侧补回去？** 补回去等价于再乘一个 $128/255$ 的系数。请结合 AIPP 的输入是 UINT8（§6 的模型输入一行）说明：程序能接触到的只有那块 UINT8 缓冲区，在整数上乘一个小于 $1$ 的系数会得到什么。由此说明**为什么归一化必须由 AIPP 或模型自身完成，而不能留给应用侧**。

**3. 把通道顺序弄反一次。** 把 §5.3 的 `cv2.cvtColor` 那一行去掉，直接把 BGR 送进去，两版都重跑。请回答：① 检出结果怎么变——是完全没有目标，还是框还在但类别或置信度不对？② **程序有没有报任何错？** ③ 由此说明结论 ① 里那句**有些错误不会当场报出来**指的是什么。做完请把这一行改回去。

**4. 换一张长宽比不同的图片。** 换一张更接近正方形或更细长的图片重跑一遍。请回答：① `SCALE`、`PAD_X`、`PAD_Y` 各自怎么变？② 画出来的框还落在物体上吗？③ 若把 §5.3 的 letterbox 改成直接拉伸到 $640 \times 640$，框会偏成什么样、为什么。

## 14. 🤔 思考题

**1.** 本实验把三步转换交给了 AIPP。另外两种做法是：在设备上显式调用媒体数据处理算子（§2.1），或者在导出 ONNX 时把这三步写进模型的图里。三种做法各自的代价是什么？**请从这三步发生在编译期还是运行期、要不要写进应用程序、参数能不能在运行期改这三个角度分析。**

**2.** 静态 AIPP 把参数固化在 `.om` 里。什么样的部署场景下这一点不可接受，必须改用动态 AIPP？**请结合 §3.3 的五条约束说明动态 AIPP 的代价**，并回答：既然动态 AIPP 更灵活，为什么本实验仍然选静态。

**3.** §10 测到的预处理收益里，有一部分来自主机少算了，有一部分来自少传了四分之三的数据。**这两部分能不能分开量出来？** 请设计一个测量方案。提示：可以再写一个版本，主机照常做三步转换，但把结果转成 UINT8 再传——这样计算没省、传输省了。

**4.** 本实验的后处理是在主机上做的 NMS。它有没有可能搬到设备上？**请分别从它算什么与它的数据在哪两个角度分析**：NMS 的排序与两两比较适合 AI Core 吗？若把它留在主机，D2H 要传的是整个候选框张量——有没有办法让模型只把留下来的那几十个框传回来？

## 15. 📌 本实验小结

<!-- markdown 版本（保留备用；如需切回，删除本注释标记并注释掉下方 HTML 表格）
| 要点 | 内容 |
| --- | --- |
| 部署链路 | PyTorch 权重 → ONNX（选 ATC 支持的 opset）→ <code>.om</code>（ATC，输入名与形状要给对）→ AscendCL 程序 |
| 阶段分解 | 端到端要拆成预处理、推理、后处理三段再动手；三段分别计时，才能把某一段的变化与总时间的变化区分开 |
| 受控对照 | 比较两种做法之前，先把其余条件按住：同一份输入、同一份程序、同样的后处理。做不到这一点，测出的差值归不到任何一个原因上 |
| 检测的两端 | 输入端 letterbox 保比例补灰边（注意 OpenCV 读出来是 BGR）；输出端从 (1, 84, 8400) 解框、再用 NMS 去重 |
| 预处理的三步 | 转 FLOAT、除以 255、HWC 换成 CHW。放在主机是逐像素的循环加四倍的传输；放进 AIPP 则两项都省下 |
| AIPP 的代价 | 它是模型的一部分，做的事占设备的时间，出现在推理那一段。<strong>净收益要看端到端</strong> |
| 两条路的分工 | 媒体数据处理算子在运行期显式调用，适合按图变化的处理，工程代价是额外的库与内存管理；AIPP 在转换时固化，适合固定不变的处理，工程代价接近于零 |
| 编译期与运行期 | 参数不变的事写进 <code>.om</code>，按图变化的事留在运行期；这条分界决定用 AIPP 还是用算子 |
| 结果核对 | 两版的类别与取整后的框坐标应逐个相同（置信度只作观察量）；把框画回原图是整条链路是否正确的最终判据 |
| 平台边界 | A3 支持 VPC/JPEGD/JPEGE/PNGD/VDEC，<strong>不支持 VENC</strong>、Camera、NVR 与音频 |
-->

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
<thead>
<tr>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">要点</th>
<th style="border:1px solid #cccccc; padding:6px 12px; text-align:left; background-color:#f5f5f5;">内容</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">部署链路</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">PyTorch 权重 → ONNX（选 ATC 支持的 opset）→ <code>.om</code>（ATC，输入名与形状要给对）→ AscendCL 程序</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">阶段分解</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">端到端要拆成预处理、推理、后处理三段再动手；三段分别计时，才能把某一段的变化与总时间的变化区分开</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">受控对照</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">比较两种做法之前，先把其余条件按住：同一份输入、同一份程序、同样的后处理。做不到这一点，测出的差值归不到任何一个原因上</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">检测的两端</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">输入端 letterbox 保比例补灰边（注意 OpenCV 读出来是 BGR）；输出端从 (1, 84, 8400) 解框、再用 NMS 去重</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">预处理的三步</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">转 FLOAT、除以 255、HWC 换成 CHW。放在主机是逐像素的循环加四倍的传输；放进 AIPP 则两项都省下</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">AIPP 的代价</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">它是模型的一部分，做的事占设备的时间，出现在推理那一段。<strong>净收益要看端到端</strong></td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两条路的分工</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">媒体数据处理算子在运行期显式调用，适合按图变化的处理，工程代价是额外的库与内存管理；AIPP 在转换时固化，适合固定不变的处理，工程代价接近于零</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">编译期与运行期</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">参数不变的事写进 <code>.om</code>，按图变化的事留在运行期；这条分界决定用 AIPP 还是用算子</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">结果核对</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">两版的类别与取整后的框坐标应逐个相同（置信度只作观察量）；把框画回原图是整条链路是否正确的最终判据</td>
</tr>
<tr>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">平台边界</td>
<td style="border:1px solid #cccccc; padding:6px 12px; text-align:left;">A3 支持 VPC/JPEGD/JPEGE/PNGD/VDEC，<strong>不支持 VENC</strong>、Camera、NVR 与音频</td>
</tr>
</tbody>
</table>